In [ ]:
#---Main Text---#
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import itertools

In [ ]:

## Code for “Activation energy asymmetry organizes temperature responses of transcriptional-translation feedback loop oscillators”


##

# Figure 1: Schematic illustration of the temperature-dependent TTFL oscillator.
# Figure 2: Temperature-dependent changes in rising ratio and amplitude in the Jeong-Kim model.
# Figure 3: Phase sensitivity, period sensitivity, and limit-cycle analyses.
# Figure 4: Analysis of waveform, phase, and period responses across multiple TTFL oscillator models.
# Figure 5: Coupled-oscillator simulations and analysis of phase sensitivity, period sensitivity, and the infinitesimal phase response curve (iPRC).




# The code uses the following packages:

#- NumPy
#- SciPy
#- Matplotlib




## Numerical resolution

#The default simulation time step was set to `step_size = 0.01`. A finer time step of `step_size = 0.001` was used for quantities requiring higher temporal resolution. Specifically, this finer resolution was used for the protein rising-ratio calculations in all oscillator models, the phase-sensitivity calculations for the Repressilator and Pentilator models, and the iPRC calculations. In these analyses, the total number of simulation steps was increased accordingly to ensure a sufficiently long simulation duration.
#For Figure 2B, the rising ratio and oscillation amplitude were evaluated together from the same simulations. Because the rising-ratio calculation requires the finer temporal resolution, all simulations used to generate Figure 2B were therefore performed with `step_size = 0.001`. The oscillation amplitude itself was essentially insensitive to the difference between `step_size = 0.01` and `0.001`.


In [ ]:
# Figure 1
### This is merely a cartoon figure. It may differ slightly from the actual illustration in the manuscript. ###

In [ ]:
# Set up: Defining the transcription function f(R) (Jeong et al., 2022)

AT = 0.35           # total activator
KA = 10**(-4)       # Normalized activation rate constant
KB = 10**(-5)       # Normalized binding rate constant
KS = 10**(-5)       # Normalized sequestration rate constant
KD = 10**(-1)       # Normalized displacement rate constant

sigma = (KS*KD)/(KA*KB)

step_count = 30000
step_size = 0.01 # To measure the rising ratio of protein oscillation, the calculation was performed with a reduced step size of 0.001 to increase resolution.
time = np.linspace(0, step_count * step_size, step_count + 1)

K_a = KA*AT
K_b = KB*AT
K_s = KS*AT
K_d = KD*AT

#----The equation in reference----#

# Concentration of activator
def A(x):
  return ((AT - x - K_s) + np.sqrt((AT - x - K_s)**2 + 4 * AT * K_s))/2

# Concentration of nuclear repressor
def R(x):
  return x - (AT - A(x))

# Transcription rate
def g(x):
  return ((K_s + sigma*K_a + R(x))*(A(x)/K_a))/((K_s + sigma*K_a + sigma*R(x)) + (K_s + sigma*K_a + R(x))*(A(x)/K_a) + (K_s + K_a + R(x))*(R(x)/K_b)*(A(x)/K_a))


# In simulations, it is more efficient to simplify the equation as $f(R)$.
C1 = AT - K_s
C2 = 4*AT*(K_s)
C3 = K_s + sigma*(K_a)
D1 = C3+C1-AT
D2 = C3+sigma*(C1-AT)
D3 = (K_s)+(K_a)+(C1-AT)
D4 = D3*(C1-AT)+(C2/4)
D5 = (C2/4)*(D3)+(C2/4)*(C1-AT)
D6 = D3*(C1-AT)-(C2/4)
G1 = D1
G2 = C2/4
F1 = (sigma)*(K_a)+D1+(D4/K_b)
F2 = (D1-((sigma)*(K_a))+(D6/K_b))
F3 = D2*(K_a)+(C2/4)+(D5/K_b)

# $f(R)$ is exactly the same equation as $g(x)$ (transcription rate).
def f(x):
 return (G1 * (np.sqrt((C1-x)**2 + C2) + (C1-x)) + 2*G2) / (F1 * (np.sqrt((C1-x)**2 + C2)) + F2 * (C1-x)+ 2*F3)


np.random.seed(5)
initial_value = np.random.uniform(0, 1, 3)  # initial value (M_0, Rc_0, R_0)

hot_M = np.zeros(step_count + 1)
hot_Rc = np.zeros(step_count + 1)
hot_R = np.zeros(step_count + 1)

cool_M = np.zeros(step_count + 1)
cool_Rc = np.zeros(step_count + 1)
cool_R = np.zeros(step_count + 1)

hot_M[0] = initial_value[0]
hot_Rc[0] = initial_value[1]
hot_R[0] = initial_value[2]

cool_M[0] = initial_value[0]
cool_Rc[0] = initial_value[1]
cool_R[0] = initial_value[2]

hot_beta = 0.01     # inverse of high temperature (beta)
cool_beta = 0.40    # inverse of low temperature (beta)
Es = 0.4            # activation energy of synthesis reaction
Ed = 0.1            # activation energy of degradation reaction

def hot_g(M, Rc, R, h):
  k1 = np.exp(-Es * hot_beta)
  k2 = np.exp(-Ed * hot_beta)
  return np.array([M + (k1*f(R) - k2*M)*h,
                   Rc + (k1*M - k2*Rc)*h,
                   R + (k1*Rc - k2*R)*h])

def cool_g(M, Rc, R, h):
  k1 = np.exp(-Es * cool_beta)
  k2 = np.exp(-Ed * cool_beta)
  return np.array([M + (k1*f(R) - k2*M)*h,
                   Rc + (k1*M - k2*Rc)*h,
                   R + (k1*Rc - k2*R)*h])


for i in range(step_count) :
 hot_M[i+1] = hot_g(hot_M[i], hot_Rc[i], hot_R[i], step_size)[0]
 hot_Rc[i+1] = hot_g(hot_M[i], hot_Rc[i], hot_R[i], step_size)[1]
 hot_R[i+1] = hot_g(hot_M[i], hot_Rc[i], hot_R[i], step_size)[2]

 cool_M[i+1] = cool_g(cool_M[i], cool_Rc[i], cool_R[i], step_size)[0]
 cool_Rc[i+1] = cool_g(cool_M[i], cool_Rc[i], cool_R[i], step_size)[1]
 cool_R[i+1] = cool_g(cool_M[i], cool_Rc[i], cool_R[i], step_size)[2]


plt.figure(figsize=(8, 6))
plt.plot(time[int(0.97*step_count):], hot_R[int(0.97*step_count):], 'red')
plt.plot(time[int(0.97*step_count):], cool_R[int(0.97*step_count):], 'blue')

plt.xlabel('time', fontsize=12)
plt.ylabel('Normalized Protein level', fontsize=12)
plt.title('Figure 1A-(ii)', fontsize=16)
plt.show()

In [ ]:
# Figure 2
### In this figure, we calculated the rising ratio and amplitude of the Jeong-Kim oscillator while varying the temperature.

In [ ]:
## We calculated the reference distance and reference height to clarify the criteria for quantifying the rising ratio and period.
## In fact, single-cell oscillators converge to a stable limit cycle, leading to clear, stable oscillations that may render complex algorithms unnecessary.
## However, once coupling is introduced, complex phenomena such as biperiodic or chaotic oscillations can emerge; therefore, we established a robust oscillation-detection algorithm in advance.

def ref_g(M, Rc, R, h):
  return np.array([M + (f(R) - M)*h,
                   Rc + (M - Rc)*h,
                   R + (Rc - R)*h])

ref_M = np.zeros(step_count + 1)
ref_Rc = np.zeros(step_count + 1)
ref_R = np.zeros(step_count + 1)

ref_M[0] = initial_value[0]
ref_Rc[0] = initial_value[1]
ref_R[0] =  initial_value[2]


for i in range(step_count) :
 ref_M[i+1] = ref_g(ref_M[i], ref_Rc[i], ref_R[i], step_size)[0]
 ref_Rc[i+1] = ref_g(ref_M[i], ref_Rc[i], ref_R[i], step_size)[1]
 ref_R[i+1] = ref_g(ref_M[i], ref_Rc[i], ref_R[i], step_size)[2]

ref_peaks, _ = find_peaks(ref_R[int(0.5*step_count):])
ref_valley, _ = find_peaks(-ref_R[int(0.5*step_count):])

ref_period = np.mean(np.diff(ref_peaks)) * step_size
ref_distance = ref_period

ref_amplitude = np.max(ref_R[int(0.5*step_count):]) - np.min(ref_R[int(0.5*step_count):])
ref_height = ref_amplitude

print("Reference distance : ", ref_distance)
print("Reference height : ", ref_height)



def symmetry(signal, tol = 0.1 * (ref_distance / step_size)):

 peak, _ = find_peaks(signal[int(step_count/2):])
 valley, _ = find_peaks(-signal[int(step_count/2):])

 peak_diff = np.diff(peak)
 valley_diff = np.diff(valley)

 if len(peak) < 2 :
  return None


 elif np.allclose(peak_diff, peak_diff[0], atol=tol) == False:
  return None


 elif np.allclose(valley_diff, valley_diff[0], atol=tol) == False:
  return None


 elif abs(signal[int(0.5 * step_count):][peak[0]] - signal[int(0.5 * step_count):][peak[len(peak)-1]]) > 0.05 * ref_height :
  print("Damping")
  return None

 else :
  if peak[0] > valley[0]:
   return (peak[0] - valley[0])/(valley[1] - valley[0])

  elif peak[0] < valley[0]:
   return (peak[1] - valley[0])/(valley[1] - valley[0])



def frequency(signal, tol = 0.1 * (ref_distance / step_size)) :

 peak,_ = find_peaks(signal[int(step_count/2):])
 peak_diff = np.diff(peak)

 if len(peak) < 2 or len(peak_diff) < 2 :
   print("Not enough peaks found to determine frequency.")
   return None

 elif  np.allclose(peak_diff, peak_diff[0], atol=tol) == False:
   print("Biperiodic oscillation")
   return None

 elif abs(signal[int(0.5 * step_count):][peak[0]] - signal[int(0.5 * step_count):][peak[len(peak)-1]]) > 0.05 * ref_height :
   print("Damping")
   return None

 else :
   return 1 / (np.mean(peak_diff) * step_size)



def period(signal, tol = 0.1 * (ref_distance / step_size)) :

  peak,_ = find_peaks(signal[int(step_count/2):])
  peak_diff = np.diff(peak)

  if len(peak) < 2 or len(peak_diff) < 2 :
   print("Not enough peaks found to determine frequency.")
   return None

  elif  np.allclose(peak_diff, peak_diff[0], atol=tol) == False:
   print("Biperiodic oscillation")
   return None

  elif abs(signal[int(0.5 * step_count):][peak[0]] - signal[int(0.5 * step_count):][peak[len(peak)-1]]) > 0.05 * ref_height :
   print("Damping")
   return None

  else :
   return np.mean(peak_diff) * step_size


beta_range = np.linspace(0.1, 0.5, 50)
inverse_beta_range = 1 / beta_range


#--------------- Calculating the rising ratio ---------------#
def symmetry_calculation(Es, Ed, beta):

 M = np.zeros(step_count + 1)
 Rc = np.zeros(step_count + 1)
 R = np.zeros(step_count + 1)

 np.random.seed(5)
 initial_value = np.random.uniform(0, 1, (3))

 M[0] = initial_value[0]
 Rc[0] = initial_value[1]
 R[0] = initial_value[2]

 for i in range(step_count) :

    k1 = np.exp(-beta * Es)
    k2 = np.exp(-beta * Ed)
    M[i+1] = M[i] + (k1 * f(R[i]) - k2 * M[i]) * step_size
    Rc[i+1] = Rc[i] + (k1 * M[i] - k2 * Rc[i]) * step_size
    R[i+1] = R[i] + (k1 * Rc[i] - k2 * R[i]) * step_size

 return symmetry(R)



#--------------- Calculating the oscillation period ---------------#
def period_calculation(Es, Ed, beta):

 M = np.zeros(step_count + 1)
 Rc = np.zeros(step_count + 1)
 R = np.zeros(step_count + 1)

 np.random.seed(5)
 initial_value = np.random.uniform(0, 1, (3))

 M[0] = initial_value[0]
 Rc[0] = initial_value[1]
 R[0] = initial_value[2]

 for i in range(step_count) :

    k1 = np.exp(-beta * Es)
    k2 = np.exp(-beta * Ed)
    M[i+1] = M[i] + (k1 * f(R[i]) - k2 * M[i]) * step_size
    Rc[i+1] = Rc[i] + (k1 * M[i] - k2 * Rc[i]) * step_size
    R[i+1] = R[i] + (k1 * Rc[i] - k2 * R[i]) * step_size

 return period(R)



#--------------- Calculating the amplitude ---------------#
def amplitude_calculation(Es, Ed, beta):

 M = np.zeros(step_count + 1)
 Rc = np.zeros(step_count + 1)
 R = np.zeros(step_count + 1)

 np.random.seed(5)
 initial_value = np.random.uniform(0, 1, (3))

 M[0] = initial_value[0]
 Rc[0] = initial_value[1]
 R[0] = initial_value[2]

 for i in range(step_count) :

    k1 = np.exp(-beta * Es)
    k2 = np.exp(-beta * Ed)
    M[i+1] = M[i] + (k1 * f(R[i]) - k2 * M[i]) * step_size
    Rc[i+1] = Rc[i] + (k1 * M[i] - k2 * Rc[i]) * step_size
    R[i+1] = R[i] + (k1 * Rc[i] - k2 * R[i]) * step_size

 return np.max(R[(int(0.5*step_count)):]) - np.min(R[(int(0.5*step_count)):])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

params_list = [(0.4, 0.1), (0.4, 0.4), (0.1, 0.4)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for Es, Ed in params_list:
    amplitude_list = []
    symmetry_list = []

    for beta in beta_range:
        amplitude_list.append(amplitude_calculation(Es, Ed, beta))
        symmetry_list.append(symmetry_calculation(Es, Ed, beta))

    amplitude_array = np.array(amplitude_list)
    symmetry_array = np.array(symmetry_list) * 100

    axes[0].scatter(inverse_beta_range, symmetry_array, s=15)
    axes[0].plot(inverse_beta_range, symmetry_array, label=f'Es={Es}, Ed={Ed}')

    axes[1].scatter(inverse_beta_range, amplitude_array, s=15)
    axes[1].plot(inverse_beta_range, amplitude_array, label=f'Es={Es}, Ed={Ed}')

    axes[2].scatter(symmetry_array, amplitude_array, s=15)
    axes[2].plot(symmetry_array, amplitude_array, label=f'Es={Es}, Ed={Ed}')

axes[0].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[0].set_ylabel("Rising ratio (%)", fontsize=12)
axes[0].set_title("Rising ratio (%) vs Temperature, Fig2B-(i)")

axes[1].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].set_title("Amplitude vs Temperature, Fig2B-(ii)")

axes[2].set_xlabel("Rising ratio (%)", fontsize=12)
axes[2].set_ylabel("Amplitude", fontsize=12)
axes[2].set_title("Rising ratio (%) vs Amplitude, Fig2B-(iii)")

for ax in axes:
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
hot_M1 = np.zeros(step_count + 1)
hot_Rc1 = np.zeros(step_count + 1)
hot_R1 = np.zeros(step_count + 1)

cool_M1 = np.zeros(step_count + 1)
cool_Rc1 = np.zeros(step_count + 1)
cool_R1 = np.zeros(step_count + 1)

hot_M1[0] = initial_value[0]
hot_Rc1[0] = initial_value[1]
hot_R1[0] = initial_value[2]

cool_M1[0] = initial_value[0]
cool_Rc1[0] = initial_value[1]
cool_R1[0] = initial_value[2]

hot_beta = 0.01
cool_beta = 0.40

# Waveform change with E_s = E_d #

def hot_g1(M, Rc, R, h):
  Es = 0.4
  Ed = 0.4
  return np.array([M + (np.exp(-Es * hot_beta)*f(R) - np.exp(-Ed * hot_beta)*M)*h,
                   Rc + (np.exp(-Es * hot_beta)*M - np.exp(-Ed * hot_beta)*Rc)*h,
                   R + (np.exp(-Es * hot_beta)*Rc - np.exp(-Ed * hot_beta)*R)*h])

def cool_g1(M, Rc, R, h):
  Es = 0.4
  Ed = 0.4
  return np.array([M + (np.exp(-Es * cool_beta)*f(R) - np.exp(-Ed * cool_beta)*M)*h,
                   Rc + (np.exp(-Es * cool_beta)*M - np.exp(-Ed * cool_beta)*Rc)*h,
                   R + (np.exp(-Es * cool_beta)*Rc - np.exp(-Ed * cool_beta)*R)*h])

for i in range(step_count) :
 hot_M1[i+1] = hot_g1(hot_M1[i], hot_Rc1[i], hot_R1[i], step_size)[0]
 hot_Rc1[i+1] = hot_g1(hot_M1[i], hot_Rc1[i], hot_R1[i], step_size)[1]
 hot_R1[i+1] = hot_g1(hot_M1[i], hot_Rc1[i], hot_R1[i], step_size)[2]

 cool_M1[i+1] = cool_g1(cool_M1[i], cool_Rc1[i], cool_R1[i], step_size)[0]
 cool_Rc1[i+1] = cool_g1(cool_M1[i], cool_Rc1[i], cool_R1[i], step_size)[1]
 cool_R1[i+1] = cool_g1(cool_M1[i], cool_Rc1[i], cool_R1[i], step_size)[2]


hot_M2 = np.zeros(step_count + 1)
hot_Rc2 = np.zeros(step_count + 1)
hot_R2 = np.zeros(step_count + 1)

cool_M2 = np.zeros(step_count + 1)
cool_Rc2 = np.zeros(step_count + 1)
cool_R2 = np.zeros(step_count + 1)

hot_M2[0] = initial_value[0]
hot_Rc2[0] = initial_value[1]
hot_R2[0] = initial_value[2]

cool_M2[0] = initial_value[0]
cool_Rc2[0] = initial_value[1]
cool_R2[0] = initial_value[2]


# Waveform change with E_s < E_d #

def hot_g2(M, Rc, R, h):
  Es = 0.1
  Ed = 0.4
  return np.array([M + (np.exp(-Es * hot_beta)*f(R) - np.exp(-Ed * hot_beta)*M)*h,
                   Rc + (np.exp(-Es * hot_beta)*M - np.exp(-Ed * hot_beta)*Rc)*h,
                   R + (np.exp(-Es * hot_beta)*Rc - np.exp(-Ed * hot_beta)*R)*h])

def cool_g2(M, Rc, R, h):
  Es = 0.1
  Ed = 0.4
  return np.array([M + (np.exp(-Es * cool_beta)*f(R) - np.exp(-Ed * cool_beta)*M)*h,
                   Rc + (np.exp(-Es * cool_beta)*M - np.exp(-Ed * cool_beta)*Rc)*h,
                   R + (np.exp(-Es * cool_beta)*Rc - np.exp(-Ed * cool_beta)*R)*h])

for i in range(step_count) :
 hot_M2[i+1] = hot_g2(hot_M2[i], hot_Rc2[i], hot_R2[i], step_size)[0]
 hot_Rc2[i+1] = hot_g2(hot_M2[i], hot_Rc2[i], hot_R2[i], step_size)[1]
 hot_R2[i+1] = hot_g2(hot_M2[i], hot_Rc2[i], hot_R2[i], step_size)[2]

 cool_M2[i+1] = cool_g2(cool_M2[i], cool_Rc2[i], cool_R2[i], step_size)[0]
 cool_Rc2[i+1] = cool_g2(cool_M2[i], cool_Rc2[i], cool_R2[i], step_size)[1]
 cool_R2[i+1] = cool_g2(cool_M2[i], cool_Rc2[i], cool_R2[i], step_size)[2]



plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(time[int(0.9*step_count):], hot_R[int(0.9*step_count):], 'red')
plt.plot(time[int(0.9*step_count):], cool_R[int(0.9*step_count):], 'blue')
plt.title('Figure 2B (iv) E_s > E_d')


plt.subplot(2, 2, 2)
plt.plot(time[int(0.9*step_count):], hot_R1[int(0.9*step_count):], 'red')
plt.plot(time[int(0.9*step_count):], cool_R1[int(0.9*step_count):], 'blue')
plt.title('Figure 2B (v) E_s > E_d')


plt.subplot(2, 2, 3)
plt.plot(time[int(0.9*step_count):], hot_R2[int(0.9*step_count):], 'red')
plt.plot(time[int(0.9*step_count):], cool_R2[int(0.9*step_count):], 'blue')
plt.title('Figure 2B (vi) E_s > E_d')

plt.show()

In [ ]:
# Figure 3
### In this figure, we calculated the phase sensitivity and period sensitivity while varying the temperature.

In [ ]:
def hot_update_uncoupled(M, Rc, R, Es_value, Ed_value, hot_temp):
  h = step_size
  k1 = np.exp(-Es_value * hot_temp)
  k2 = np.exp(-Ed_value * hot_temp)
  return np.array([ M + (k1 * f(R) - k2 * M) * h, Rc + (k1 * M - k2 * Rc) * h, R + (k1 * Rc - k2 * R) * h ])


def cool_update_uncoupled(M, Rc, R, Es_value, Ed_value, cool_temp):
  h = step_size
  k1 = np.exp(-Es_value * cool_temp)
  k2 = np.exp(-Ed_value * cool_temp)
  return np.array([ M + (k1 * f(R) - k2 * M) * h, Rc + (k1 * M - k2 * Rc) * h, R + (k1 * Rc - k2 * R) * h ])


def hot_particle_simulation_uncoupled(Es_value, Ed_value, hot_temp) :
  particle = np.zeros((step_count + 1, 3))
  np.random.seed(3)
  particle[0, :] = np.random.uniform(0, 1, (3))

  for i in range(step_count):
    particle[i+1, :] = hot_update_uncoupled(particle[i, 0], particle[i, 1], particle[i, 2], Es_value, Ed_value, hot_temp)

  return particle


def cool_particle_simulation_uncoupled(Es_value, Ed_value, cool_temp) :
  particle = np.zeros((step_count + 1, 3))
  np.random.seed(3)
  particle[0, :] = np.random.uniform(0, 1, (3))

  for i in range(step_count):
    particle[i+1, :] = cool_update_uncoupled(particle[i, 0], particle[i, 1], particle[i, 2], Es_value, Ed_value, cool_temp)

  return particle

In [ ]:
Es_value = 0.4
degradation_energy = np.linspace(0.0, 0.4, 10)

hot_temp = 0.01
cool_temp = 0.40

degradation_energy_list = []
phase_sensitivity_list = []
period_sensitivity_list = []
amplitude_sensitivity_list = []

pulse_strength = 0.25
pulse_number = 30

for Ed in degradation_energy:

  Es = Es_value

  uncoupled_hot_signal = hot_particle_simulation_uncoupled(Es, Ed, hot_temp)
  uncoupled_cool_signal = cool_particle_simulation_uncoupled(Es, Ed, cool_temp)

  uncoupled_hot_period = period(uncoupled_hot_signal[:, 2])
  uncoupled_cool_period = period(uncoupled_cool_signal[:, 2])
  uncoupled_hot_amplitude = amplitude_calculation(Es, Ed, hot_beta)
  uncoupled_cool_amplitude = amplitude_calculation(Es, Ed, cool_beta)

  if uncoupled_hot_period == None  or  uncoupled_cool_period == None:
    print(f"No valid period values")
    continue


## Calculating the phase shift ##

  cool_peaks,_ = find_peaks(uncoupled_cool_signal[int(0.5 * step_count):, 2])

  pulse_time = int((pulse_strength) * (uncoupled_cool_period / step_size))

  starting_point = int(cool_peaks[1] + 0.5 * step_count)
  ending_point = int(cool_peaks[2] + 0.5 * step_count)

  pulse_points = np.linspace(starting_point, ending_point, pulse_number)

  phase_shift_list = []

  for pulse_point in pulse_points:

    perturbed_particle = np.zeros(((step_count - int(pulse_point) + 1), 3))
    perturbed_particle[0, :] = uncoupled_cool_signal[int(pulse_point), :]

    for i in range(pulse_time + 1):

      perturbed_particle[i+1, :] = hot_update_uncoupled(perturbed_particle[i, 0], perturbed_particle[i, 1], perturbed_particle[i, 2], Es, Ed, hot_temp)

    for k in range(step_count - int(pulse_point) - pulse_time - 1):

      perturbed_particle[k+2+pulse_time, :] = cool_update_uncoupled(perturbed_particle[k+1+pulse_time, 0], perturbed_particle[k+1+pulse_time, 1], perturbed_particle[k+1+pulse_time, 2], Es, Ed, cool_temp)


    rescaled_signal = uncoupled_cool_signal[int(pulse_point):, 2]
    rescaled_time = time[int(pulse_point):]

    unperturbed_peak,_ = find_peaks(rescaled_signal)
    length_of_peak = len(unperturbed_peak)

    if length_of_peak < 3 :
      print("Not enough peak")
      continue

    one_peak_point = unperturbed_peak[length_of_peak - 2]
    other_peak_point = unperturbed_peak[length_of_peak - 1]

    perturbed_peak,_ = find_peaks(perturbed_particle[:, 2])
    perturbed_valley,_ = find_peaks(-perturbed_particle[:, 2])
    valley,_ = find_peaks(-rescaled_signal)

    valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
    peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
    valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

    if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1 :

      phase_shift = 0

    else :

      phase_shift = 24 * ( (valley_point[0] - valley_perturbed_point[0]) / int((uncoupled_cool_period / step_size)) )
      # 24h unit

    phase_shift_list.append(phase_shift)

  maximal_shift = max(phase_shift_list)
  minimal_shift = min(phase_shift_list)
  mean_shift = np.mean(phase_shift_list)

  phase_sensitivity = (maximal_shift - minimal_shift)/2

  degradation_energy_list.append(Ed)
  phase_sensitivity_list.append(phase_sensitivity)
  period_sensitivity_list.append(uncoupled_cool_period / uncoupled_hot_period)
  amplitude_sensitivity_list.append(uncoupled_cool_amplitude / uncoupled_hot_amplitude)



fig, ax1 = plt.subplots(figsize=(7, 5))

color1 = 'tab:red'
ax1.plot(degradation_energy_list, phase_sensitivity_list,
         color=color1, marker='s', linestyle='--',
         label='Phase sensitivity')
ax1.set_xlabel('Degradation energy')
ax1.set_ylabel('Phase sensitivity', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)


ax2 = ax1.twinx()

color2 = 'tab:blue'
ax2.plot(degradation_energy_list, period_sensitivity_list,
         color=color2, marker='o',
         label='Period sensitivity')
ax2.set_ylabel('Period sensitivity', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)


ax1.invert_xaxis()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

from matplotlib.ticker import MaxNLocator

ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))

plt.title('Figure 3A-(iii)')
plt.tight_layout()
plt.show()

print(period_sensitivity_list)
print(amplitude_sensitivity_list)

In [ ]:
plt.figure(figsize=(14, 10))

uncoupled_hot_signal_sym = hot_particle_simulation_uncoupled(0.4, 0.4, 0.01)
uncoupled_cool_signal_sym = cool_particle_simulation_uncoupled(0.4, 0.4, 0.40)

plt.subplot(2, 2, 1)
plt.plot(uncoupled_hot_signal_sym[int(step_count/2):, 0], uncoupled_hot_signal_sym[int(step_count/2):, 2], color='r')
plt.plot(uncoupled_cool_signal_sym[int(step_count/2):, 0], uncoupled_cool_signal_sym[int(step_count/2):, 2], color='b')
plt.xlabel('M', fontsize=12)
plt.ylabel('R', fontsize=12)
plt.title('Figure 3B-(i): Ed = Es', fontsize=14)
plt.xlim(0.05, 0.80)
plt.ylim(0.28, 0.48)


uncoupled_hot_signal_asym = hot_particle_simulation_uncoupled(0.4, 0.1, 0.01)
uncoupled_cool_signal_asym = cool_particle_simulation_uncoupled(0.4, 0.1, 0.40)

plt.subplot(2, 2, 2)
plt.plot(uncoupled_hot_signal_asym[int(step_count/2):, 0], uncoupled_hot_signal_asym[int(step_count/2):, 2], color='r')
plt.plot(uncoupled_cool_signal_asym[int(step_count/2):, 0], uncoupled_cool_signal_asym[int(step_count/2):, 2], color='b')
plt.xlabel('M', fontsize=12)
plt.ylabel('R', fontsize=12)
plt.title('Figure 3B-(i): Ed = Es', fontsize=14)
plt.xlim(0.05, 0.80)
plt.ylim(0.28, 0.48)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=False)

param_cases = [(0.4, 0.4), (0.4, 0.1)]

for ax, (Es, Ed) in zip(axes, param_cases):

    uncoupled_hot_signal_test = hot_particle_simulation_uncoupled(Es, Ed, hot_temp)
    uncoupled_cool_signal_test = cool_particle_simulation_uncoupled(Es, Ed, cool_temp)

    uncoupled_hot_period_test = period(uncoupled_hot_signal_test[:, 0])
    uncoupled_cool_period_test = period(uncoupled_cool_signal_test[:, 0])

    cool_peaks,_ = find_peaks(uncoupled_cool_signal_test[int(0.5 * step_count):, 0])
    cool_valleys,_ = find_peaks(-uncoupled_cool_signal_test[int(0.5 * step_count):, 0])

    hot_peaks,_ = find_peaks(uncoupled_hot_signal_test[int(0.5 * step_count):, 0])
    hot_valleys,_ = find_peaks(-uncoupled_hot_signal_test[int(0.5 * step_count):, 0])

    pulse_time = int((pulse_strength) * (uncoupled_cool_period_test / step_size))

    starting_point = int(cool_peaks[1] + 0.5 * step_count)
    ending_point = int(cool_peaks[2] + 0.5 * step_count)

    pulse_points = np.linspace(starting_point, ending_point, pulse_number)
    pulse_points = [pulse_points[2]]

    for pulse_point in pulse_points:

        perturbed_particle = np.zeros(((step_count - int(pulse_point) + 1), 3))
        perturbed_particle[0, :] = uncoupled_cool_signal_test[int(pulse_point), :]

        # pulse (hot)
        for i in range(pulse_time + 1):
            perturbed_particle[i+1, :] = hot_update_uncoupled(
                perturbed_particle[i, 0],
                perturbed_particle[i, 1],
                perturbed_particle[i, 2],
                Es, Ed, hot_temp
            )

        # recovery (cool)
        for k in range(step_count - int(pulse_point) - pulse_time - 1):
            perturbed_particle[k+2+pulse_time, :] = cool_update_uncoupled(
                perturbed_particle[k+1+pulse_time, 0],
                perturbed_particle[k+1+pulse_time, 1],
                perturbed_particle[k+1+pulse_time, 2],
                Es, Ed, cool_temp
            )

        rescaled_time = time[int(pulse_point):]

        # ---- plotting ----
        ax.plot(
            time[int(pulse_point)-200:int(pulse_point+1200)],
            uncoupled_cool_signal_test[int(pulse_point)-200:int(pulse_point+1200), 2],
            'black', label='Unperturbed'
        )

        ax.plot(
            rescaled_time[0:int(pulse_time)],
            perturbed_particle[0:int(pulse_time), 2],
            color='gray', linestyle='--'
        )

        ax.plot(
            rescaled_time[int(pulse_time):1200],
            perturbed_particle[int(pulse_time):1200, 2],
            color='gray', linestyle='--', label='Perturbed'
        )

    ax.set_title(f'Es={Es}, Ed={Ed}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Figure 4
### In this figure, we calculated the rising ratio, amplitude, phase, and period sensitivity across various types of TTFL oscillators.

In [ ]:
# Repressilator / We used the same analytical tools as those applied to the Jeong-Kim model.

step_size = 0.01  # When calculating the rising ratio and phase sensitivity, the step size was set to 0.001 to increase the resolution. This is because these physical quantities exhibit very subtle differences among data points compared to the amplitude or period sensitivity.
t_end = 1000
step_count = int(t_end / step_size)
time = np.linspace(0, t_end, step_count + 1)

n_param = 2
gamma0_param = 0.0005
gamma_ref = 0.5
kb_param = 1.0 / 1600.0
km_ref = 0.006
kp_ref = 0.0012
tau_ref = 0.12
scaling_factor = 100

def repressilator_update(m1, m2, m3, p1, p2, p3, Es, Ed, temp):
    gamma_eff = scaling_factor * gamma_ref * np.exp(-Es * temp)
    tau_eff   = scaling_factor * tau_ref   * np.exp(-Es * temp)
    km_eff    = scaling_factor * km_ref    * np.exp(-Ed * temp)
    kp_eff    = scaling_factor * kp_ref    * np.exp(-Ed * temp)

    dm1dt = -km_eff * m1 + (gamma_eff / (1.0 + kb_param * (p3**n_param))) + gamma0_param * scaling_factor * np.exp(-Es * temp)
    dm2dt = -km_eff * m2 + (gamma_eff / (1.0 + kb_param * (p1**n_param))) + gamma0_param * scaling_factor * np.exp(-Es * temp)
    dm3dt = -km_eff * m3 + (gamma_eff / (1.0 + kb_param * (p2**n_param))) + gamma0_param * scaling_factor * np.exp(-Es * temp)

    dp1dt = -kp_eff * p1 + tau_eff * m1
    dp2dt = -kp_eff * p2 + tau_eff * m2
    dp3dt = -kp_eff * p3 + tau_eff * m3

    m1_next = m1 + dm1dt * step_size
    m2_next = m2 + dm2dt * step_size
    m3_next = m3 + dm3dt * step_size

    p1_next = p1 + dp1dt * step_size
    p2_next = p2 + dp2dt * step_size
    p3_next = p3 + dp3dt * step_size

    return m1_next, m2_next, m3_next, p1_next, p2_next, p3_next

def run_repressilator_simulation(Es, Ed, temp):
    signal = np.zeros((step_count + 1, 6))

    np.random.seed(5)
    initial_value = np.random.uniform(5, 25, 6)

    signal[0, :] = initial_value

    for i in range(step_count):
        m1, m2, m3, p1, p2, p3 = signal[i, :]
        m1_n, m2_n, m3_n, p1_n, p2_n, p3_n = repressilator_update(m1, m2, m3, p1, p2, p3, Es, Ed, temp)
        signal[i+1, :] = [m1_n, m2_n, m3_n, p1_n, p2_n, p3_n]

    return signal

def hot_particle_simulation_uncoupled(Es, Ed, temp):
    return run_repressilator_simulation(Es, Ed, temp)

def cool_particle_simulation_uncoupled(Es, Ed, temp):
    return run_repressilator_simulation(Es, Ed, temp)

def hot_update_uncoupled(m1, m2, m3, p1, p2, p3, Es, Ed, temp):
    return repressilator_update(m1, m2, m3, p1, p2, p3, Es, Ed, temp)

def cool_update_uncoupled(m1, m2, m3, p1, p2, p3, Es, Ed, temp):
    return repressilator_update(m1, m2, m3, p1, p2, p3, Es, Ed, temp)



def symmetry(signal):

 peak, _ = find_peaks(signal[int(step_count/2):])
 valley, _ = find_peaks(-signal[int(step_count/2):])

 peak_diff = np.diff(peak)
 valley_diff = np.diff(valley)

 if len(peak) < 2 :
  return None

 else :
  if peak[0] > valley[0]:
   return (peak[0] - valley[0])/(valley[1] - valley[0])

  elif peak[0] < valley[0]:
   return (peak[1] - valley[0])/(valley[1] - valley[0])


def frequency(signal) :

 peak,_ = find_peaks(signal[int(step_count/2):])
 peak_diff = np.diff(peak)

 if len(peak) < 2 or len(peak_diff) < 2 :
   print("Not enough peaks found to determine frequency.")
   return None

 else :
   return 1 / (np.mean(peak_diff) * step_size)


def period(signal) :

  peak,_ = find_peaks(signal[int(step_count/2):])
  peak_diff = np.diff(peak)

  if len(peak) < 2 or len(peak_diff) < 2 :
   print("Not enough peaks found to determine frequency.")
   return None

  else :
   return np.mean(peak_diff) * step_size


beta_range = np.linspace(0.1, 0.5, 20)
inverse_beta_range = 1 / beta_range

def symmetry_calculation(Es, Ed, beta):
 sig = run_repressilator_simulation(Es, Ed, beta)
 return symmetry(sig[:, 5])

def period_calculation(Es, Ed, beta):
 sig = run_repressilator_simulation(Es, Ed, beta)
 return period(sig[:, 5])

def amplitude_calculation(Es, Ed, beta):
 sig = run_repressilator_simulation(Es, Ed, beta)
 M = sig[:, 5]
 return np.max(M[(int(0.5*step_count)):]) - np.min(M[(int(0.5*step_count)):])


params_list = [(0.4, 0.1), (0.4, 0.4), (0.1, 0.4)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for Es, Ed in params_list:
    amplitude_list = []
    symmetry_list = []


    for beta in beta_range:
        amplitude_list.append(amplitude_calculation(Es, Ed, beta))
        symmetry_list.append(symmetry_calculation(Es, Ed, beta))


    amplitude_array = np.array(amplitude_list)
    symmetry_array = np.array(symmetry_list) * 100

    axes[0].scatter(inverse_beta_range, symmetry_array, s=15)
    axes[0].plot(inverse_beta_range, symmetry_array, label=f'Es={Es}, Ed={Ed}')

    axes[1].scatter(inverse_beta_range, amplitude_array, s=15)
    axes[1].plot(inverse_beta_range, amplitude_array, label=f'Es={Es}, Ed={Ed}')

    axes[2].scatter(symmetry_array, amplitude_array, s=15)
    axes[2].plot(symmetry_array, amplitude_array, label=f'Es={Es}, Ed={Ed}')

axes[0].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[0].set_ylabel("Rising ratio (%)", fontsize=12)
axes[0].yaxis.set_major_locator(MaxNLocator(nbins=5))
axes[0].set_title("Rising ratio (%) vs Temperature, Repressilator")

axes[1].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].set_title("Amplitude vs Temperature, Repressilator")

axes[2].set_xlabel("Rising ratio (%)", fontsize=12)
axes[2].set_ylabel("Amplitude", fontsize=12)
axes[2].set_title("Rising ratio (%) vs Amplitude, Repressilator")

for ax in axes:
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
Es_value = 0.4
degradation_energy = np.linspace(0.0, 0.4, 10)

hot_temp = 0.01
cool_temp = 0.40

degradation_energy_list = []
phase_sensitivity_list = []
period_sensitivity_list = []
amplitude_sensitivity_list = []

pulse_strength = 0.25
pulse_number = 30

plt.plot(time, run_repressilator_simulation(Es_value, degradation_energy[0], hot_temp)[:, 5], label='Hot')
plt.show()
for Ed in degradation_energy:

  Es = Es_value

  uncoupled_hot_signal = hot_particle_simulation_uncoupled(Es, Ed, hot_temp)
  uncoupled_cool_signal = cool_particle_simulation_uncoupled(Es, Ed, cool_temp)

  uncoupled_hot_period = period(uncoupled_hot_signal[:, 5])
  uncoupled_cool_period = period(uncoupled_cool_signal[:, 5])
  uncoupled_hot_amplitude = amplitude_calculation(Es, Ed, hot_temp)
  uncoupled_cool_amplitude = amplitude_calculation(Es, Ed, cool_temp)

  if uncoupled_hot_period == None  or  uncoupled_cool_period == None:
    print(f"No valid period values")
    continue

  cool_peaks,_ = find_peaks(uncoupled_cool_signal[int(0.5 * step_count):, 5])

  pulse_time = int((pulse_strength) * (uncoupled_cool_period / step_size))

  starting_point = int(cool_peaks[1] + 0.5 * step_count)
  ending_point = int(cool_peaks[2] + 0.5 * step_count)

  pulse_points = np.linspace(starting_point, ending_point, pulse_number)

  phase_shift_list = []

  for pulse_point in pulse_points:

    perturbed_particle = np.zeros(((step_count - int(pulse_point) + 2), 6))
    perturbed_particle[0, :] = uncoupled_cool_signal[int(pulse_point), :]

    for i in range(pulse_time + 1):

      m1_n, m2_n, m3_n, p1_n, p2_n, p3_n = hot_update_uncoupled(
          perturbed_particle[i, 0], perturbed_particle[i, 1], perturbed_particle[i, 2],
          perturbed_particle[i, 3], perturbed_particle[i, 4], perturbed_particle[i, 5],
          Es, Ed, hot_temp
      )
      perturbed_particle[i+1, :] = [m1_n, m2_n, m3_n, p1_n, p2_n, p3_n]

    for k in range(step_count - int(pulse_point) - pulse_time - 1):

      m1_n, m2_n, m3_n, p1_n, p2_n, p3_n = cool_update_uncoupled(
          perturbed_particle[k+1+pulse_time, 0], perturbed_particle[k+1+pulse_time, 1], perturbed_particle[k+1+pulse_time, 2],
          perturbed_particle[k+1+pulse_time, 3], perturbed_particle[k+1+pulse_time, 4], perturbed_particle[k+1+pulse_time, 5],
          Es, Ed, cool_temp
      )
      perturbed_particle[k+2+pulse_time, :] = [m1_n, m2_n, m3_n, p1_n, p2_n, p3_n]


    rescaled_signal = uncoupled_cool_signal[int(pulse_point):, 5]
    rescaled_time = time[int(pulse_point):]

    unperturbed_peak,_ = find_peaks(rescaled_signal)
    length_of_peak = len(unperturbed_peak)

    if length_of_peak < 3 :
      print("Not enough peak")
      continue

    one_peak_point = unperturbed_peak[length_of_peak - 2]
    other_peak_point = unperturbed_peak[length_of_peak - 1]

    perturbed_peak,_ = find_peaks(perturbed_particle[:, 5])
    perturbed_valley,_ = find_peaks(-perturbed_particle[:, 5])
    valley,_ = find_peaks(-rescaled_signal)

    valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
    peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
    valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

    if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1 :

      phase_shift = 0

    else :

      phase_shift = 24 * ( (valley_point[0] - valley_perturbed_point[0]) / int((uncoupled_cool_period / step_size)) )
      # 24h unit

    phase_shift_list.append(phase_shift)

  if len(phase_shift_list) == 0:
    continue

  maximal_shift = max(phase_shift_list)
  minimal_shift = min(phase_shift_list)
  mean_shift = np.mean(phase_shift_list)

  phase_sensitivity = (maximal_shift - minimal_shift)/2

  degradation_energy_list.append(Ed)
  phase_sensitivity_list.append(phase_sensitivity)
  period_sensitivity_list.append(uncoupled_cool_period / uncoupled_hot_period)
  amplitude_sensitivity_list.append(uncoupled_cool_amplitude / uncoupled_hot_amplitude)


fig, ax1 = plt.subplots(figsize=(7, 5))

color1 = 'tab:red'
ax1.plot(degradation_energy_list, phase_sensitivity_list,
         color=color1, marker='s', linestyle='--',
         label='Phase sensitivity')
ax1.set_xlabel('Degradation energy')
ax1.set_ylabel('Phase sensitivity', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)


ax2 = ax1.twinx()

color2 = 'tab:blue'
ax2.plot(degradation_energy_list, period_sensitivity_list,
         color=color2, marker='o',
         label='Period sensitivity')
ax2.set_ylabel('Period sensitivity', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)


ax1.invert_xaxis()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))

plt.title('Phase and period sensitivity: Repressilator')
plt.tight_layout()
plt.show()

In [ ]:
# Pentilator / We used the same analytical tools as those applied to the Jeong-Kim model.

step_size = 0.01  # When calculating the rising ratio and phase sensitivity, the step size was set to 0.001 to increase the resolution. This is because these physical quantities exhibit very subtle differences among data points compared to the amplitude or period sensitivity.
t_end = 1000

step_size = 0.001
t_end = 1000
step_count = int(t_end / step_size)
time = np.linspace(0, t_end, step_count + 1)

n_param = 2
gamma0_param = 0.0005
gamma_ref = 0.5
kb_param = 1.0 / 1600.0
km_ref = 0.006
kp_ref = 0.0012
tau_ref = 0.12
scaling_parameter = 200

def pentilator_update(m, p, Es, Ed, temp):

    gamma_eff = scaling_parameter * gamma_ref * np.exp(-Es * temp)
    tau_eff   = scaling_parameter * tau_ref   * np.exp(-Es * temp)
    km_eff    = scaling_parameter * km_ref    * np.exp(-Ed * temp)
    kp_eff    = scaling_parameter * kp_ref    * np.exp(-Ed * temp)

    # j = 5, 1, 2, 3, 4 => [4, 0, 1, 2, 3]
    p_j = np.roll(p, 1)

    dmdt = -km_eff * m + (gamma_eff / (1.0 + kb_param * (p_j**n_param))) + gamma0_param * scaling_parameter * np.exp(-Es * temp)
    dpdt = -kp_eff * p + tau_eff * m

    m_next = m + dmdt * step_size
    p_next = p + dpdt * step_size

    return m_next, p_next

def run_pentilator_simulation(Es, Ed, temp):

    signal = np.zeros((step_count + 1, 10))

    np.random.seed(5)
    initial_value = np.random.uniform(5, 25, 10)

    signal[0, :] = initial_value

    for i in range(step_count):
        m = signal[i, :5]
        p = signal[i, 5:]
        m_n, p_n = pentilator_update(m, p, Es, Ed, temp)
        signal[i+1, :5] = m_n
        signal[i+1, 5:] = p_n

    return signal

def hot_particle_simulation_uncoupled(Es, Ed, temp):
    return run_pentilator_simulation(Es, Ed, temp)

def cool_particle_simulation_uncoupled(Es, Ed, temp):
    return run_pentilator_simulation(Es, Ed, temp)

def hot_update_uncoupled(m, p, Es, Ed, temp):
    return pentilator_update(m, p, Es, Ed, temp)

def cool_update_uncoupled(m, p, Es, Ed, temp):
    return pentilator_update(m, p, Es, Ed, temp)


def symmetry(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    valley, _ = find_peaks(-signal[int(step_count/2):])

    if len(peak) < 2 or len(valley) < 2:
        return None

    if peak[0] > valley[0]:
        return (peak[0] - valley[0]) / (valley[1] - valley[0])
    elif peak[0] < valley[0]:
        return (peak[1] - valley[0]) / (valley[1] - valley[0])

def period(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    peak_diff = np.diff(peak)

    if len(peak) < 2 or len(peak_diff) < 2:
        return None
    else:
        return np.mean(peak_diff) * step_size


beta_range = np.linspace(0.1, 0.5, 20)
inverse_beta_range = 1 / beta_range

def symmetry_calculation(Es, Ed, beta):
    sig = run_pentilator_simulation(Es, Ed, beta)
    return symmetry(sig[:, 9])

def period_calculation(Es, Ed, beta):
    sig = run_pentilator_simulation(Es, Ed, beta)
    return period(sig[:, 9])

def amplitude_calculation(Es, Ed, beta):
    sig = run_pentilator_simulation(Es, Ed, beta)
    M = sig[:, 9]
    return np.max(M[(int(0.5*step_count)):]) - np.min(M[(int(0.5*step_count)):])



params_list = [(0.4, 0.1), (0.4, 0.4), (0.1, 0.4)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for Es, Ed in params_list:
    amplitude_list = []
    symmetry_list = []

    for beta in beta_range:
        amp = amplitude_calculation(Es, Ed, beta)
        sym = symmetry_calculation(Es, Ed, beta)

        amplitude_list.append(amp if amp is not None else 0)
        symmetry_list.append(sym if sym is not None else 0)

    amplitude_array = np.array(amplitude_list)
    symmetry_array = np.array(symmetry_list) * 100

    axes[0].scatter(inverse_beta_range, symmetry_array, s=15)
    axes[0].plot(inverse_beta_range, symmetry_array, label=f'Es={Es}, Ed={Ed}')

    axes[1].scatter(inverse_beta_range, amplitude_array, s=15)
    axes[1].plot(inverse_beta_range, amplitude_array, label=f'Es={Es}, Ed={Ed}')

    axes[2].scatter(symmetry_array, amplitude_array, s=15)
    axes[2].plot(symmetry_array, amplitude_array, label=f'Es={Es}, Ed={Ed}')

axes[0].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[0].set_ylabel("Rising ratio (%)", fontsize=12)
axes[0].set_title("Rising ratio (%) vs Temperature, Pentilator")

axes[1].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].set_title("Amplitude vs Temperature, Pentilator")

axes[2].set_xlabel("Rising ratio (%)", fontsize=12)
axes[2].set_ylabel("Amplitude", fontsize=12)
axes[2].set_title("Rising ratio (%) vs Amplitude, Pentilator")

for ax in axes:
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
Es_value = 0.4
degradation_energy = np.linspace(0.0, 0.4, 10)

hot_temp = 0.01
cool_temp = 0.40

degradation_energy_list = []
phase_sensitivity_list = []
period_sensitivity_list = []
amplitude_sensitivity_list = []

pulse_strength = 0.25
pulse_number = 30



for Ed in degradation_energy:
    Es = Es_value

    uncoupled_hot_signal = hot_particle_simulation_uncoupled(Es, Ed, hot_temp)
    uncoupled_cool_signal = cool_particle_simulation_uncoupled(Es, Ed, cool_temp)

    uncoupled_hot_period = period(uncoupled_hot_signal[:, 9])
    uncoupled_cool_period = period(uncoupled_cool_signal[:, 9])
    uncoupled_hot_amplitude = amplitude_calculation(Es, Ed, hot_temp)
    uncoupled_cool_amplitude = amplitude_calculation(Es, Ed, cool_temp)


    if uncoupled_hot_period is None or uncoupled_cool_period is None:
        print(f"No valid period values for Ed={Ed}")
        continue

    cool_peaks, _ = find_peaks(uncoupled_cool_signal[int(0.5 * step_count):, 9])

    if len(cool_peaks) < 3:
        continue

    pulse_time = int((pulse_strength) * (uncoupled_cool_period / step_size))

    starting_point = int(cool_peaks[1] + 0.5 * step_count)
    ending_point = int(cool_peaks[2] + 0.5 * step_count)

    pulse_points = np.linspace(starting_point, ending_point, pulse_number)
    phase_shift_list = []

    for pulse_point in pulse_points:
        perturbed_particle = np.zeros(((step_count - int(pulse_point) + 2), 10))
        perturbed_particle[0, :] = uncoupled_cool_signal[int(pulse_point), :]

        for i in range(pulse_time + 1):
            m_n, p_n = hot_update_uncoupled(
                perturbed_particle[i, :5], perturbed_particle[i, 5:],
                Es, Ed, hot_temp
            )
            perturbed_particle[i+1, :5] = m_n
            perturbed_particle[i+1, 5:] = p_n

        for k in range(step_count - int(pulse_point) - pulse_time - 1):
            m_n, p_n = cool_update_uncoupled(
                perturbed_particle[k+1+pulse_time, :5], perturbed_particle[k+1+pulse_time, 5:],
                Es, Ed, cool_temp
            )
            perturbed_particle[k+2+pulse_time, :5] = m_n
            perturbed_particle[k+2+pulse_time, 5:] = p_n

        rescaled_signal = uncoupled_cool_signal[int(pulse_point):, 9]

        unperturbed_peak, _ = find_peaks(rescaled_signal)
        length_of_peak = len(unperturbed_peak)

        if length_of_peak < 3:
            continue

        one_peak_point = unperturbed_peak[length_of_peak - 2]
        other_peak_point = unperturbed_peak[length_of_peak - 1]

        perturbed_peak, _ = find_peaks(perturbed_particle[:, 9])
        perturbed_valley, _ = find_peaks(-perturbed_particle[:, 9])
        valley, _ = find_peaks(-rescaled_signal)

        valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
        peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
        valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

        if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1:
            phase_shift = 0
        else:
            phase_shift = 24 * ((valley_point[0] - valley_perturbed_point[0]) / int((uncoupled_cool_period / step_size)))

        phase_shift_list.append(phase_shift)

    if len(phase_shift_list) == 0:
        continue

    maximal_shift = max(phase_shift_list)
    minimal_shift = min(phase_shift_list)

    phase_sensitivity = (maximal_shift - minimal_shift) / 2

    degradation_energy_list.append(Ed)
    phase_sensitivity_list.append(phase_sensitivity)
    period_sensitivity_list.append(uncoupled_cool_period / uncoupled_hot_period)
    amplitude_sensitivity_list.append(uncoupled_cool_amplitude / uncoupled_hot_amplitude)


fig, ax1 = plt.subplots(figsize=(7, 5))

color1 = 'tab:red'
ax1.plot(degradation_energy_list, phase_sensitivity_list,
         color=color1, marker='s', linestyle='--',
         label='Phase sensitivity')
ax1.set_xlabel('Degradation energy')
ax1.set_ylabel('Phase sensitivity', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()

color2 = 'tab:blue'
ax2.plot(degradation_energy_list, period_sensitivity_list,
         color=color2, marker='o',
         label='Period sensitivity')
ax2.set_ylabel('Period sensitivity', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.invert_xaxis()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))

plt.title('Phase and period sensitivity: Pentilator')
plt.tight_layout()
plt.show()

In [ ]:
# Kim-Forger model / We used the same analytical tools as those applied to the Jeong-Kim model.

step_size = 0.01  # When calculating the rising ratio, the step size was set to 0.001 to increase the resolution. This is because these physical quantities exhibit very subtle differences among data points compared to the amplitude, period sensitivity, or phase sensitivity.
t_end = 1000

step_size = 0.001
t_end = 1000
step_count = int(t_end / step_size)
time = np.linspace(0, t_end, step_count + 1)

k1_ref = 1.0
k2_ref = 1.0
k3_ref = 1.0
k4_ref = 0.16
k5_ref = 0.29
k6_ref = 0.3

k7 = 0.6
k8 = 1e-5


def kim_forger_update(x, y, z, Es, Ed, temp):

    term1 = k7 - k8 - z
    f_z = (term1 + np.sqrt(term1**2 + 4.0 * k7 * k8)) / (2.0 * k7)

    scale_s = np.exp(-Es * temp)
    scale_d = np.exp(-Ed * temp)

    k1 = k1_ref * scale_s
    k2 = k2_ref * scale_s
    k3 = k3_ref * scale_s

    k4 = k4_ref * scale_d
    k5 = k5_ref * scale_d
    k6 = k6_ref * scale_d

    dxdt = k1 * f_z - k4 * x
    dydt = k2 * x - k5 * y
    dzdt = k3 * y - k6 * z  ## MM type: dzdt = k3 * y - k6 * ((3.8*z) / 1+z)

    x_next = x + dxdt * step_size
    y_next = y + dydt * step_size
    z_next = z + dzdt * step_size

    return x_next, y_next, z_next

def run_kim_forger_simulation(Es, Ed, temp):

    signal = np.zeros((step_count + 1, 3))

    np.random.seed(5)
    initial_value = np.random.uniform(0.1, 0.2, 3)

    signal[0, :] = initial_value

    for i in range(step_count):
        x, y, z = signal[i, :]
        x_n, y_n, z_n = kim_forger_update(x, y, z, Es, Ed, temp)
        signal[i+1, :] = [x_n, y_n, z_n]

    return signal

def hot_particle_simulation_uncoupled(Es, Ed, temp):
    return run_kim_forger_simulation(Es, Ed, temp)

def cool_particle_simulation_uncoupled(Es, Ed, temp):
    return run_kim_forger_simulation(Es, Ed, temp)

def hot_update_uncoupled(x, y, z, Es, Ed, temp):
    return kim_forger_update(x, y, z, Es, Ed, temp)

def cool_update_uncoupled(x, y, z, Es, Ed, temp):
    return kim_forger_update(x, y, z, Es, Ed, temp)



def symmetry(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    valley, _ = find_peaks(-signal[int(step_count/2):])

    if len(peak) < 2 or len(valley) < 2:
        return None

    if peak[0] > valley[0]:
        return (peak[0] - valley[0]) / (valley[1] - valley[0])
    elif peak[0] < valley[0]:
        return (peak[1] - valley[0]) / (valley[1] - valley[0])

def frequency(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    peak_diff = np.diff(peak)

    if len(peak) < 2 or len(peak_diff) < 2:
        print("Not enough peaks found to determine frequency.")
        return None
    else:
        return 1 / (np.mean(peak_diff) * step_size)

def period(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    peak_diff = np.diff(peak)

    if len(peak) < 2 or len(peak_diff) < 2:
        print("Not enough peaks found to determine period.")
        return None
    else:
        return np.mean(peak_diff) * step_size


beta_range = np.linspace(0.1, 0.5, 20)
inverse_beta_range = 1 / beta_range

def symmetry_calculation(Es, Ed, beta):
    sig = run_kim_forger_simulation(Es, Ed, beta)
    return symmetry(sig[:, 2])

def period_calculation(Es, Ed, beta):
    sig = run_kim_forger_simulation(Es, Ed, beta)
    return period(sig[:, 2])

def amplitude_calculation(Es, Ed, beta):
    sig = run_kim_forger_simulation(Es, Ed, beta)
    M = sig[:, 2]
    return np.max(M[(int(0.5*step_count)):]) - np.min(M[(int(0.5*step_count)):])


params_list = [(0.4, 0.1), (0.4, 0.4), (0.1, 0.4)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for Es, Ed in params_list:
    amplitude_list = []
    symmetry_list = []

    for beta in beta_range:
        amp = amplitude_calculation(Es, Ed, beta)
        sym = symmetry_calculation(Es, Ed, beta)

        amplitude_list.append(amp if amp is not None else 0)
        symmetry_list.append(sym if sym is not None else 0)

    amplitude_array = np.array(amplitude_list)
    symmetry_array = np.array(symmetry_list) * 100

    axes[0].scatter(inverse_beta_range, symmetry_array, s=15)
    axes[0].plot(inverse_beta_range, symmetry_array, label=f'Es={Es}, Ed={Ed}')

    axes[1].scatter(inverse_beta_range, amplitude_array, s=15)
    axes[1].plot(inverse_beta_range, amplitude_array, label=f'Es={Es}, Ed={Ed}')

    axes[2].scatter(symmetry_array, amplitude_array, s=15)
    axes[2].plot(symmetry_array, amplitude_array, label=f'Es={Es}, Ed={Ed}')

axes[0].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[0].set_ylabel("Rising ratio (%)", fontsize=12)
axes[0].set_title("Rising ratio (%) vs Temperature, Kim-Forger")

axes[1].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].set_title("Amplitude vs Temperature, Kim-Forger")

axes[2].set_xlabel("Rising ratio (%)", fontsize=12)
axes[2].set_ylabel("Amplitude", fontsize=12)
axes[2].set_title("Rising ratio (%) vs Amplitude, Kim-Forger")

for ax in axes:
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
Es_value = 0.4
degradation_energy = np.linspace(0.0, 0.4, 10)

hot_temp = 0.01
cool_temp = 0.40

degradation_energy_list = []
phase_sensitivity_list = []
period_sensitivity_list = []
amplitude_sensitivity_list = []

pulse_strength = 0.25
pulse_number = 30

for Ed in degradation_energy:

    Es = Es_value

    uncoupled_hot_signal = hot_particle_simulation_uncoupled(Es, Ed, hot_temp)
    uncoupled_cool_signal = cool_particle_simulation_uncoupled(Es, Ed, cool_temp)

    uncoupled_hot_period = period(uncoupled_hot_signal[:, 2])
    uncoupled_cool_period = period(uncoupled_cool_signal[:, 2])
    uncoupled_hot_amplitude = amplitude_calculation(Es, Ed, hot_temp)
    uncoupled_cool_amplitude = amplitude_calculation(Es, Ed, cool_temp)

    if uncoupled_hot_period is None or uncoupled_cool_period is None:
        print(f"No valid period values for Ed={Ed}")
        continue

    cool_peaks, _ = find_peaks(uncoupled_cool_signal[int(0.5 * step_count):, 2])

    if len(cool_peaks) < 3:
        continue

    pulse_time = int((pulse_strength) * (uncoupled_cool_period / step_size))

    starting_point = int(cool_peaks[1] + 0.5 * step_count)
    ending_point = int(cool_peaks[2] + 0.5 * step_count)

    pulse_points = np.linspace(starting_point, ending_point, pulse_number)

    phase_shift_list = []

    for pulse_point in pulse_points:

        perturbed_particle = np.zeros(((step_count - int(pulse_point) + 2), 3))
        perturbed_particle[0, :] = uncoupled_cool_signal[int(pulse_point), :]

        for i in range(pulse_time + 1):
            x_n, y_n, z_n = hot_update_uncoupled(
                perturbed_particle[i, 0], perturbed_particle[i, 1], perturbed_particle[i, 2],
                Es, Ed, hot_temp
            )
            perturbed_particle[i+1, :] = [x_n, y_n, z_n]

        for k in range(step_count - int(pulse_point) - pulse_time - 1):
            x_n, y_n, z_n = cool_update_uncoupled(
                perturbed_particle[k+1+pulse_time, 0], perturbed_particle[k+1+pulse_time, 1], perturbed_particle[k+1+pulse_time, 2],
                Es, Ed, cool_temp
            )
            perturbed_particle[k+2+pulse_time, :] = [x_n, y_n, z_n]

        rescaled_signal = uncoupled_cool_signal[int(pulse_point):, 2]

        unperturbed_peak, _ = find_peaks(rescaled_signal)
        length_of_peak = len(unperturbed_peak)

        if length_of_peak < 3:
            continue

        one_peak_point = unperturbed_peak[length_of_peak - 2]
        other_peak_point = unperturbed_peak[length_of_peak - 1]

        perturbed_peak, _ = find_peaks(perturbed_particle[:, 2])
        perturbed_valley, _ = find_peaks(-perturbed_particle[:, 2])
        valley, _ = find_peaks(-rescaled_signal)

        valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
        peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
        valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

        if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1:
            phase_shift = 0
        else:
            phase_shift = 24 * ((valley_point[0] - valley_perturbed_point[0]) / int((uncoupled_cool_period / step_size)))

        phase_shift_list.append(phase_shift)

    if len(phase_shift_list) == 0:
        continue

    maximal_shift = max(phase_shift_list)
    minimal_shift = min(phase_shift_list)

    phase_sensitivity = (maximal_shift - minimal_shift) / 2

    degradation_energy_list.append(Ed)
    phase_sensitivity_list.append(phase_sensitivity)
    period_sensitivity_list.append(uncoupled_cool_period / uncoupled_hot_period)
    amplitude_sensitivity_list.append(uncoupled_cool_amplitude / uncoupled_hot_amplitude)


fig, ax1 = plt.subplots(figsize=(7, 5))

color1 = 'tab:red'
ax1.plot(degradation_energy_list, phase_sensitivity_list,
         color=color1, marker='s', linestyle='--',
         label='Phase sensitivity')
ax1.set_xlabel('Degradation energy')
ax1.set_ylabel('Phase sensitivity', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()

color2 = 'tab:blue'
ax2.plot(degradation_energy_list, period_sensitivity_list,
         color=color2, marker='o',
         label='Period sensitivity')
ax2.set_ylabel('Period sensitivity', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.invert_xaxis()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))

plt.title('Phase and period sensitivity: Kim-Forger')
plt.tight_layout()
plt.show()

In [ ]:
# Goodwin model / We used the same analytical tools as those applied to the Jeong-Kim model.

step_size = 0.01  # When calculating the rising ratio, the step size was set to 0.001 to increase the resolution. This is because these physical quantities exhibit very subtle differences among data points compared to the amplitude, period sensitivity, or phase sensitivity.
t_end = 1000
step_count = int(t_end / step_size)
time = np.linspace(0, t_end, step_count + 1)

k1_ref = 1.0
k2_ref = 1.0
k3_ref = 1.0
k4_ref = 0.16
k5_ref = 0.29
k6_ref = 0.3
k_7 = 1
n = 15

def goodwin_update(x, y, z, Es, Ed, temp):

    f_z = 1/(1 + (z/k_7)**n)

    scale_s = np.exp(-Es * temp)
    scale_d = np.exp(-Ed * temp)

    k1 = k1_ref * scale_s
    k2 = k2_ref * scale_s
    k3 = k3_ref * scale_s

    k4 = k4_ref * scale_d
    k5 = k5_ref * scale_d
    k6 = k6_ref * scale_d

    dxdt = k1 * f_z - k4 * x
    dydt = k2 * x - k5 * y
    dzdt = k3 * y - k6 * z    ## MM type: dzdt = k3 * y - k6 * ((3.8*z) / 1+z)

    x_next = x + dxdt * step_size
    y_next = y + dydt * step_size
    z_next = z + dzdt * step_size

    return x_next, y_next, z_next

def run_goodwin_simulation(Es, Ed, temp):

    signal = np.zeros((step_count + 1, 3))

    np.random.seed(5)
    initial_value = np.random.uniform(0.1, 0.2, 3)

    signal[0, :] = initial_value

    for i in range(step_count):
        x, y, z = signal[i, :]
        x_n, y_n, z_n = goodwin_update(x, y, z, Es, Ed, temp)
        signal[i+1, :] = [x_n, y_n, z_n]

    return signal

def hot_particle_simulation_uncoupled(Es, Ed, temp):
    return run_goodwin_simulation(Es, Ed, temp)

def cool_particle_simulation_uncoupled(Es, Ed, temp):
    return run_goodwin_simulation(Es, Ed, temp)

def hot_update_uncoupled(x, y, z, Es, Ed, temp):
    return goodwin_update(x, y, z, Es, Ed, temp)

def cool_update_uncoupled(x, y, z, Es, Ed, temp):
    return goodwin_update(x, y, z, Es, Ed, temp)



def symmetry(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    valley, _ = find_peaks(-signal[int(step_count/2):])

    if len(peak) < 2 or len(valley) < 2:
        return None

    if peak[0] > valley[0]:
        return (peak[0] - valley[0]) / (valley[1] - valley[0])
    elif peak[0] < valley[0]:
        return (peak[1] - valley[0]) / (valley[1] - valley[0])

def frequency(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    peak_diff = np.diff(peak)

    if len(peak) < 2 or len(peak_diff) < 2:
        print("Not enough peaks found to determine frequency.")
        return None
    else:
        return 1 / (np.mean(peak_diff) * step_size)

def period(signal):
    peak, _ = find_peaks(signal[int(step_count/2):])
    peak_diff = np.diff(peak)

    if len(peak) < 2 or len(peak_diff) < 2:
        print("Not enough peaks found to determine period.")
        return None
    else:
        return np.mean(peak_diff) * step_size


beta_range = np.linspace(0.1, 0.5, 20)
inverse_beta_range = 1 / beta_range

def symmetry_calculation(Es, Ed, beta):
    sig = run_goodwin_simulation(Es, Ed, beta)
    return symmetry(sig[:, 2])

def period_calculation(Es, Ed, beta):
    sig = run_goodwin_simulation(Es, Ed, beta)
    return period(sig[:, 2])

def amplitude_calculation(Es, Ed, beta):
    sig = run_goodwin_simulation(Es, Ed, beta)
    M = sig[:, 2]
    return np.max(M[(int(0.5*step_count)):]) - np.min(M[(int(0.5*step_count)):])


params_list = [(0.4, 0.1), (0.4, 0.4), (0.1, 0.4)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for Es, Ed in params_list:
    amplitude_list = []
    symmetry_list = []

    for beta in beta_range:
        amp = amplitude_calculation(Es, Ed, beta)
        sym = symmetry_calculation(Es, Ed, beta)

        amplitude_list.append(amp if amp is not None else 0)
        symmetry_list.append(sym if sym is not None else 0)

    amplitude_array = np.array(amplitude_list)
    symmetry_array = np.array(symmetry_list) * 100

    axes[0].scatter(inverse_beta_range, symmetry_array, s=15)
    axes[0].plot(inverse_beta_range, symmetry_array, label=f'Es={Es}, Ed={Ed}')

    axes[1].scatter(inverse_beta_range, amplitude_array, s=15)
    axes[1].plot(inverse_beta_range, amplitude_array, label=f'Es={Es}, Ed={Ed}')

    axes[2].scatter(symmetry_array, amplitude_array, s=15)
    axes[2].plot(symmetry_array, amplitude_array, label=f'Es={Es}, Ed={Ed}')

axes[0].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[0].set_ylabel("Rising ratio (%)", fontsize=12)
axes[0].set_title("Rising ratio (%) vs Temperature, Goodwin")

axes[1].set_xlabel("Temperature (1/beta)", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].set_title("Amplitude vs Temperature, Goodwin")

axes[2].set_xlabel("Rising ratio (%)", fontsize=12)
axes[2].set_ylabel("Amplitude", fontsize=12)
axes[2].set_title("Rising ratio (%) vs Amplitude, Goodwin")

for ax in axes:
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
Es_value = 0.4
degradation_energy = np.linspace(0.0, 0.4, 10)

hot_temp = 0.01
cool_temp = 0.40

degradation_energy_list = []
phase_sensitivity_list = []
period_sensitivity_list = []
amplitude_sensitivity_list = []

pulse_strength = 0.25
pulse_number = 30

for Ed in degradation_energy:

    Es = Es_value

    uncoupled_hot_signal = hot_particle_simulation_uncoupled(Es, Ed, hot_temp)
    uncoupled_cool_signal = cool_particle_simulation_uncoupled(Es, Ed, cool_temp)

    uncoupled_hot_period = period(uncoupled_hot_signal[:, 2])
    uncoupled_cool_period = period(uncoupled_cool_signal[:, 2])
    uncoupled_hot_amplitude = amplitude_calculation(Es, Ed, hot_temp)
    uncoupled_cool_amplitude = amplitude_calculation(Es, Ed, cool_temp)

    if uncoupled_hot_period is None or uncoupled_cool_period is None:
        print(f"No valid period values for Ed={Ed}")
        continue

    cool_peaks, _ = find_peaks(uncoupled_cool_signal[int(0.5 * step_count):, 2])

    if len(cool_peaks) < 3:
        continue

    pulse_time = int((pulse_strength) * (uncoupled_cool_period / step_size))

    starting_point = int(cool_peaks[1] + 0.5 * step_count)
    ending_point = int(cool_peaks[2] + 0.5 * step_count)

    pulse_points = np.linspace(starting_point, ending_point, pulse_number)

    phase_shift_list = []

    for pulse_point in pulse_points:

        perturbed_particle = np.zeros(((step_count - int(pulse_point) + 2), 3))
        perturbed_particle[0, :] = uncoupled_cool_signal[int(pulse_point), :]

        for i in range(pulse_time + 1):
            x_n, y_n, z_n = hot_update_uncoupled(
                perturbed_particle[i, 0], perturbed_particle[i, 1], perturbed_particle[i, 2],
                Es, Ed, hot_temp
            )
            perturbed_particle[i+1, :] = [x_n, y_n, z_n]

        for k in range(step_count - int(pulse_point) - pulse_time - 1):
            x_n, y_n, z_n = cool_update_uncoupled(
                perturbed_particle[k+1+pulse_time, 0], perturbed_particle[k+1+pulse_time, 1], perturbed_particle[k+1+pulse_time, 2],
                Es, Ed, cool_temp
            )
            perturbed_particle[k+2+pulse_time, :] = [x_n, y_n, z_n]

        rescaled_signal = uncoupled_cool_signal[int(pulse_point):, 2]

        unperturbed_peak, _ = find_peaks(rescaled_signal)
        length_of_peak = len(unperturbed_peak)

        if length_of_peak < 3:
            continue

        one_peak_point = unperturbed_peak[length_of_peak - 2]
        other_peak_point = unperturbed_peak[length_of_peak - 1]

        perturbed_peak, _ = find_peaks(perturbed_particle[:, 2])
        perturbed_valley, _ = find_peaks(-perturbed_particle[:, 2])
        valley, _ = find_peaks(-rescaled_signal)

        valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
        peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
        valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

        if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1:
            phase_shift = 0
        else:
            phase_shift = 24 * ((valley_point[0] - valley_perturbed_point[0]) / int((uncoupled_cool_period / step_size)))

        phase_shift_list.append(phase_shift)

    if len(phase_shift_list) == 0:
        continue

    maximal_shift = max(phase_shift_list)
    minimal_shift = min(phase_shift_list)

    phase_sensitivity = (maximal_shift - minimal_shift) / 2

    degradation_energy_list.append(Ed)
    phase_sensitivity_list.append(phase_sensitivity)
    period_sensitivity_list.append(uncoupled_cool_period / uncoupled_hot_period)
    amplitude_sensitivity_list.append(uncoupled_cool_amplitude / uncoupled_hot_amplitude)


# 최종 플롯 출력
fig, ax1 = plt.subplots(figsize=(7, 5))

color1 = 'tab:red'
ax1.plot(degradation_energy_list, phase_sensitivity_list,
         color=color1, marker='s', linestyle='--',
         label='Phase sensitivity')
ax1.set_xlabel('Degradation energy')
ax1.set_ylabel('Phase sensitivity', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()

color2 = 'tab:blue'
ax2.plot(degradation_energy_list, period_sensitivity_list,
         color=color2, marker='o',
         label='Period sensitivity')
ax2.set_ylabel('Period sensitivity', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.invert_xaxis()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))

plt.title('Phase and period sensitivity: Goodwin')
plt.tight_layout()
plt.show()

In [ ]:
# Figure 5
### In this figure, we added the coupling term to the Jeong-Kim model and analyzed the changes in phase and period sensitivity alongside iPRC distortion.

In [ ]:
num_cell = 100
timescale = 20

np.random.seed(1)
cell_culture = np.random.normal(1, 0.1, num_cell)   # 0.1 : heterogeneity of cells

def hot_update(M, Rc, R, V, V_sum, y, coupling_input, Es_value, Ed_value, hot_temp):
  h = step_size
  k1 = np.exp(-Es_value * hot_temp)
  k2 = np.exp(-Ed_value * hot_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h + (coupling_input / num_cell) * V_sum * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
      V + timescale * (f(R) - V) * h
  ])

def cool_update(M, Rc, R, V, V_sum, y, coupling_input, Es_value, Ed_value, cool_temp):
  h = step_size
  k1 = np.exp(-Es_value * cool_temp)
  k2 = np.exp(-Ed_value * cool_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h + (coupling_input / num_cell) * V_sum * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
      V + timescale * (f(R) - V) * h
  ])

def hot_update_uncoupled(M, Rc, R, y, Es_value, Ed_value, hot_temp):
  h = step_size
  k1 = np.exp(-Es_value * hot_temp)
  k2 = np.exp(-Ed_value * hot_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
  ])

def cool_update_uncoupled(M, Rc, R, y, Es_value, Ed_value, cool_temp):
  h = step_size
  k1 = np.exp(-Es_value * cool_temp)
  k2 = np.exp(-Ed_value * cool_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
  ])

def hot_particle_simulation(coupling_input, Es_value, Ed_value, hot_temp):

    particle = np.zeros((step_count + 1, 4, num_cell))
    np.random.seed(2)
    particle[0, :, :] = np.random.uniform(0, 1, (4, num_cell))

    for i in range(step_count):
        V_sum = np.sum(particle[i, 3, :])

        for j in range(num_cell):
            y = cell_culture[j]
            particle[i + 1, :, j] = hot_update(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], particle[i, 3, j], V_sum, y, coupling_input, Es_value, Ed_value, hot_temp)

    freqs = np.array([frequency(particle[:, 2, n]) for n in range(num_cell)])
    valid_freqs = freqs[freqs != None]

    if len(valid_freqs) == 0:
        return None, False

    mean_freq = np.mean(valid_freqs)
    filtered_valid_freqs = valid_freqs[np.abs(valid_freqs - mean_freq) < 0.05 * 0.1] # 5% of heterogeneity(\sigma=0.1)

    if (len(filtered_valid_freqs) / num_cell) < 0.95: # filtering criterion
        return None, False

    else:
        return particle, True


def cool_particle_simulation(coupling_input, Es_value, Ed_value, cool_temp):
    particle = np.zeros((step_count + 1, 4, num_cell))
    np.random.seed(2)
    particle[0, :, :] = np.random.uniform(0, 1, (4, num_cell))

    for i in range(step_count):
        V_sum = np.sum(particle[i, 3, :])

        for j in range(num_cell):
            y = cell_culture[j]
            particle[i + 1, :, j] = cool_update(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], particle[i, 3, j], V_sum, y, coupling_input, Es_value, Ed_value, cool_temp)

    freqs = np.array([frequency(particle[:, 2, n]) for n in range(num_cell)])
    valid_freqs = freqs[freqs != None]

    if len(valid_freqs) == 0:
        return None, False

    mean_freq = np.mean(valid_freqs)
    filtered_valid_freqs = valid_freqs[np.abs(valid_freqs - mean_freq) < 0.05 * 0.1]

    if (len(filtered_valid_freqs) / num_cell) < 0.95:
        return None, False

    else:
        return particle, True


def hot_particle_simulation_uncoupled(Es_value, Ed_value, hot_temp) :
  particle = np.zeros((step_count + 1, 3, num_cell))
  np.random.seed(2)
  particle[0, :, :] = np.random.uniform(0, 1, (3, num_cell))

  for j in range(num_cell):
    y = cell_culture[j]

    for i in range(step_count):
      particle[i+1, :, j] = hot_update_uncoupled(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], y, Es_value, Ed_value, hot_temp)

  return particle


def cool_particle_simulation_uncoupled(Es_value, Ed_value, cool_temp) :
  particle = np.zeros((step_count + 1, 3, num_cell))
  np.random.seed(2)
  particle[0, :, :] = np.random.uniform(0, 1, (3, num_cell))

  for j in range(num_cell):
    y = cell_culture[j]

    for i in range(step_count):
      particle[i+1, :, j] = cool_update_uncoupled(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], y, Es_value, Ed_value, cool_temp)

  return particle

In [ ]:
## Calculating coupled phase sensitivity / period sensitivity

Es = 0.4
Ed_list = np.linspace(0.0, 0.4, 10)
starting_coupling = 0.0
# We confirmed that synchronization does not occur under our parameter settings when the coupling strength is 0.2 or lower. Therefore, for faster simulation, it is acceptable to run the simulation with this value set to 0.2.

pulse_strength = 0.25
pulse_number = 30

for Ed in Ed_list:

  hot_beta = 0.01
  cool_beta = 0.40

  print(f"Simulation for degradation energy : {Ed}")

  max_coupling = 0.8

  hot_coupling_strength = starting_coupling
  hot_success = False
  hot_signal = None

  while hot_coupling_strength <= max_coupling:
      hot_signal, hot_success = hot_particle_simulation(hot_coupling_strength, Es, Ed, hot_beta)

      if hot_success:
          print(f"Minimum coupling at: {hot_coupling_strength}")
          break

      else:
          print(f"Coupling does not occur at {hot_coupling_strength}")
          hot_coupling_strength += 0.05


  if not hot_success:
      print("Synchronization does not occur within this coupling range.")
      continue


  cool_coupling_strength = starting_coupling
  cool_success = False
  cool_signal = None


  while cool_coupling_strength <= max_coupling:
      cool_signal, cool_success = cool_particle_simulation(cool_coupling_strength, Es, Ed, cool_beta)

      if cool_success:
          print(f"Minimum coupling at: {cool_coupling_strength}")
          break

      else:
          print(f"Coupling does not occur at {cool_coupling_strength}")
          cool_coupling_strength += 0.05


  if not cool_success:
      print("Synchronization does not occur within this coupling range.")
      continue


  coupled_hot_signal = hot_signal
  coupled_cool_signal = cool_signal

  coupled_hot_period_list = np.array([period(coupled_hot_signal[:, 2, i]) for i in range(num_cell)])
  coupled_cool_period_list = np.array([period(coupled_cool_signal[:, 2, i]) for i in range(num_cell)])

  valid_condition = ((coupled_hot_period_list != None) & (coupled_cool_period_list != None))
  valid_indices = np.where(valid_condition)[0]

  valid_hot_period_list = coupled_hot_period_list[valid_condition]
  valid_cool_period_list = coupled_cool_period_list[valid_condition]


  if len(valid_hot_period_list) == 0 or len(valid_cool_period_list) == 0 :
    print(f"No valid period values found for both uncoupled and coupled systems")
    continue

  coupled_hot_period = np.mean(valid_hot_period_list)
  coupled_cool_period = np.mean(valid_cool_period_list)

  print(f"coupled hot period : {coupled_hot_period}")
  print(f"coupled cool period : {coupled_cool_period}")


  np.random.seed(40)
  index = int(np.random.choice(valid_indices, 1, replace=False))
  print(index)


  cool_peaks,_ = find_peaks(coupled_cool_signal[int(0.5 * step_count):, 2, index])

  pulse_time = int((pulse_strength) * (coupled_cool_period / step_size))
  pulse_number = 30

  starting_point = int(cool_peaks[1] + 0.5 * step_count)
  ending_point = int(cool_peaks[2] + 0.5 * step_count)

  pulse_points = np.linspace(starting_point, ending_point, pulse_number)
  phase_shift_list = []


  for pulse_point in pulse_points:

    perturbed_particle = np.zeros(((step_count - int(pulse_point) + 1), 4, num_cell))
    perturbed_particle[0, :, :] = coupled_cool_signal[int(pulse_point), :, :]

    for i in range(pulse_time + 1):
      V_sum = np.sum(perturbed_particle[i, 3, :])

      for j in range(num_cell):
            y = cell_culture[j]
            perturbed_particle[i+1, :, j] = hot_update(perturbed_particle[i, 0, j], perturbed_particle[i, 1, j], perturbed_particle[i, 2, j], perturbed_particle[i, 3, j], V_sum, y, hot_coupling_strength, Es, Ed, hot_beta)


    for k in range(step_count - int(pulse_point) - pulse_time - 1):
      V_sum = np.sum(perturbed_particle[k+1+pulse_time, 3, :])

      for j in range(num_cell):
       y = cell_culture[j]
       perturbed_particle[k+2+pulse_time, :, j] = cool_update(perturbed_particle[k+1+pulse_time, 0, j], perturbed_particle[k+1+pulse_time, 1, j], perturbed_particle[k+1+pulse_time, 2, j], perturbed_particle[k+1+pulse_time, 3, j], V_sum, y, cool_coupling_strength, Es, Ed, cool_beta)


    rescaled_signal = cool_signal[int(pulse_point):, 2, index]
    rescaled_time = time[int(pulse_point):]

    unperturbed_peak,_ = find_peaks(rescaled_signal)
    length_of_peak = len(unperturbed_peak)

    if length_of_peak < 3 :
      print("Not enough peak")

    one_peak_point = unperturbed_peak[length_of_peak - 2]
    other_peak_point = unperturbed_peak[length_of_peak - 1]

    perturbed_peak,_ = find_peaks(perturbed_particle[:, 2, index])
    perturbed_valley,_ = find_peaks(-perturbed_particle[:, 2, index])
    valley,_ = find_peaks(-rescaled_signal)

    valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
    peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
    valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

    if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1 :

      phase_shift = 0

    else :
      phase_shift = 24 * ( (valley_point[0] - valley_perturbed_point[0]) / int((coupled_cool_period / step_size)) )

    phase_shift_list.append(phase_shift)

  maximal_shift = max(phase_shift_list)
  minimal_shift = min(phase_shift_list)
  phase_sensitivity = (maximal_shift - minimal_shift)/2
  print(f"phase sensitivity : {phase_sensitivity}")
  print(f"period sensitivity : {coupled_cool_period / coupled_hot_period}")

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

Es = 0.4
Ed = 0.1
AT = 0.35
initial_value = [0.1, 0.1, 0.1]

beta_value = np.linspace(0.0, 0.8, 9)  # temperature
Zf_integral_list = []
positive_phase_ratio = []
symmetry_list = []

step_size = 0.001
time_end = 50000
num_iter = int(time_end / step_size)

t_cycle = np.linspace(0, time_end, num_iter)

# Transcription function is too complicated, so we may use another function with similar shape.
def trans(z, alpha, k=200):
    return 1 / (1 + np.exp(k * (z / alpha - 1))) # alpha corresponds to AT (concentration of total activator)

def derivative(z, alpha, k=200):
    term = np.exp(k * (z / alpha - 1))
    return -k / alpha * term / ((1 + term)**2)

# Jacobian matrix
def jacobian_MR(x, y, z, alpha):
    dz = derivative(z, alpha)
    n = len(z)
    J = np.zeros((3, 3, n))

    J[0, 0, :] = -k2
    J[0, 1, :] = 0
    J[0, 2, :] = k1 * dz

    J[1, 0, :] = k1
    J[1, 1, :] = -k2
    J[1, 2, :] = 0

    J[2, 0, :] = 0
    J[2, 1, :] = k1
    J[2, 2, :] = -k2

    return J


def multiple_repressor(t, w):
    x, y, z = w
    dxdt = k1 * trans(z, AT) - k2 * x
    dydt = k1 * x - k2 * y
    dzdt = k1 * y - k2 * z
    return [dxdt, dydt, dzdt]


# Backward Euler method

def backward_solver(initial_data):

    x = np.zeros(len(t_cut))
    y = np.zeros(len(t_cut))
    z = np.zeros(len(t_cut))

    x[-1], y[-1], z[-1] = initial_data

    dt_scalar = t_cut[1] - t_cut[0]

    for i in range(len(t_cut) - 1, 0, -1):

        f = -J[:, :, i].T @ np.array([x[i], y[i], z[i]])
        x[i-1] = x[i] - dt_scalar * f[0]
        y[i-1] = y[i] - dt_scalar * f[1]
        z[i-1] = z[i] - dt_scalar * f[2]

    return x, y, z

In [ ]:
# By alternating the value of beta, we can get the plots of oscillation of mRNA, iPRC, and limit cycle.

for beta in [0.01]:
    k1 = np.exp(-Es * beta)
    k2 = np.exp(-Ed * beta)

    sol = solve_ivp(
        multiple_repressor,
        [0, time_end],
        initial_value,
        t_eval=t_cycle,
        rtol=1e-6,
        atol=1e-9
    )

    x_cycle_full, y_cycle_full, z_cycle_full = sol.y

    cut = 40000
    start_idx = max(0, len(x_cycle_full) - cut)

    x_cycle = x_cycle_full[start_idx:]
    y_cycle = y_cycle_full[start_idx:]
    z_cycle = z_cycle_full[start_idx:]
    t_cut = t_cycle[start_idx:]

    dt = t_cut[1] - t_cut[0]
    J = jacobian_MR(x_cycle, y_cycle, z_cycle, AT)

    dx_dt = np.gradient(x_cycle, t_cut)
    dy_dt = np.gradient(y_cycle, t_cut)
    dz_dt = np.gradient(z_cycle, t_cut)

    Zx_raw, Zy_raw, Zz_raw = backward_solver(np.random.rand(3))

    dot_products = (
        Zx_raw * dx_dt +
        Zy_raw * dy_dt +
        Zz_raw * dz_dt
    )

    plt.plot(t_cut, dot_products)
    print(np.mean(dot_products))
    print(np.std(dot_products))
    print(np.std(dot_products) / np.sqrt(len(dot_products)))

In [ ]:
# By alternating the value of beta, we can get the plots of oscillation of mRNA, iPRC, and limit cycle.

for beta in [0.40]:
    k1 = np.exp(-Es * beta)
    k2 = np.exp(-Ed * beta)

    sol = solve_ivp(
        multiple_repressor,
        [0, time_end],
        initial_value,
        t_eval=t_cycle,
        rtol=1e-6,
        atol=1e-9
    )

    x_cycle_full, y_cycle_full, z_cycle_full = sol.y

    cut = 40000
    start_idx = max(0, len(x_cycle_full) - cut)

    x_cycle = x_cycle_full[start_idx:]
    y_cycle = y_cycle_full[start_idx:]
    z_cycle = z_cycle_full[start_idx:]
    t_cut = t_cycle[start_idx:]

    dt = t_cut[1] - t_cut[0]
    J = jacobian_MR(x_cycle, y_cycle, z_cycle, AT)

    dx_dt = np.gradient(x_cycle, t_cut)
    dy_dt = np.gradient(y_cycle, t_cut)
    dz_dt = np.gradient(z_cycle, t_cut)

    Zx_raw, Zy_raw, Zz_raw = backward_solver(np.random.rand(3))

    dot_product = (
        Zx_raw[-1] * dx_dt[-1] +
        Zy_raw[-1] * dy_dt[-1] +
        Zz_raw[-1] * dz_dt[-1]
    )
    print(dot_product)


    if abs(dot_product) < 1e-9:
        norm_factor = 1.0
    else:
        norm_factor = dot_product

    Zx = Zx_raw / norm_factor
    Zy = Zy_raw / norm_factor
    Zz = Zz_raw / norm_factor

    peaks, _ = find_peaks(-Zx)

    if len(peaks) < 4:
        print(f"beta = {beta:.3f}: Not enough peaks found (len={len(peaks)}). Skipping.")

    i0, i1 = peaks[2], peaks[3]

    Z_cycle = Zx[i0:i1]
    z_cycle_local = z_cycle[i0:i1]
    x_cycle_local = x_cycle[i0:i1]

    t_local = t_cut[i0:i1]



def plot_by_sign(x, y, ref, pos_color='red', neg_color='blue', lw=2):

    sign = np.sign(ref)
    sign[sign == 0] = 1

    change_idx = np.where(np.diff(sign) != 0)[0] + 1
    segments = np.split(np.arange(len(x)), change_idx)

    for seg in segments:
        if len(seg) < 2:
            continue

        if ref[seg[0]] > 0:
            plt.plot(x[seg], y[seg], color=pos_color, lw=lw)
        else:
            plt.plot(x[seg], y[seg], color=neg_color, lw=lw)


plt.figure(figsize=(6,4))
plot_by_sign(t_local, x_cycle_local, Z_cycle)
plt.scatter(t_local[int(len(x_cycle_local)*0.08)], x_cycle_local[int(len(x_cycle_local)*0.08)], color='black', s=20)
plt.scatter(t_local[int(len(x_cycle_local)*0.56)], x_cycle_local[int(len(x_cycle_local)*0.56)], color='black', s=20)
plt.ylim(0.05, 0.8)
plt.xlabel("Time")
plt.ylabel("mRNA")
plt.title(f"beta = {beta}, mRNA")
plt.tight_layout()
plt.savefig('exercise_mrna_Cool_v2.eps', format='eps')
plt.show()

# -------- iPRC --------
plt.figure(figsize=(6,4))
plot_by_sign(t_local, Z_cycle, Z_cycle)
plt.scatter(t_local[int(len(x_cycle_local)*0.08)], Z_cycle[int(len(x_cycle_local)*0.08)], color='black', s=20)
plt.scatter(t_local[int(len(x_cycle_local)*0.56)], Z_cycle[int(len(x_cycle_local)*0.56)], color='black', s=20)
plt.axhline(0, color='black', linestyle='--', lw=1)

plt.xlabel("Time")
plt.ylabel("iPRC")
plt.title(f"beta = {beta}, IPRC")
plt.tight_layout()
#plt.savefig('exercise_iprc.eps', format='eps')
plt.show()


# -------- Limit cycle --------
plt.figure(figsize=(6,4))
plot_by_sign(x_cycle_local, z_cycle_local, Z_cycle)
plt.scatter(x_cycle_local[int(len(x_cycle_local)*0.08)], z_cycle_local[int(len(x_cycle_local)*0.08)], color='black', s=20)
plt.scatter(x_cycle_local[int(len(x_cycle_local)*0.56)], z_cycle_local[int(len(x_cycle_local)*0.56)], color='black', s=20)
plt.xlabel("mRNA")
plt.ylabel("Protein (z)")
plt.title(f"beta = {beta}, Limit cycle")
plt.xlim(0.05, 0.8)
plt.tight_layout()
plt.show()

In [ ]:
#---Supplementary Information---#

In [ ]:
# Figure S1

In [ ]:
AT = 0.35
KA = 10**(-4)
KB = 10**(-5)
KS = 10**(-5)
KD = 10**(-1)

sigma = (KS*KD)/(KA*KB)

step_count = 30000
step_size = 0.01
time = np.linspace(0, step_count * step_size, step_count + 1)

K_a = KA*AT
K_b = KB*AT
K_s = KS*AT
K_d = KD*AT

def A(x):
  return ((AT - x - K_s) + np.sqrt((AT - x - K_s)**2 + 4 * AT * K_s))/2

def R(x):
  return x - (AT - A(x))

def g(x):
  return ((K_s + sigma*K_a + R(x))*(A(x)/K_a))/((K_s + sigma*K_a + sigma*R(x)) + (K_s + sigma*K_a + R(x))*(A(x)/K_a) + (K_s + K_a + R(x))*(R(x)/K_b)*(A(x)/K_a))


C1 = AT - K_s
C2 = 4*AT*(K_s)
C3 = K_s + sigma*(K_a)
D1 = C3+C1-AT
D2 = C3+sigma*(C1-AT)
D3 = (K_s)+(K_a)+(C1-AT)
D4 = D3*(C1-AT)+(C2/4)
D5 = (C2/4)*(D3)+(C2/4)*(C1-AT)
D6 = D3*(C1-AT)-(C2/4)
G1 = D1
G2 = C2/4
F1 = (sigma)*(K_a)+D1+(D4/K_b)
F2 = (D1-((sigma)*(K_a))+(D6/K_b))
F3 = D2*(K_a)+(C2/4)+(D5/K_b)

def f(x):
 return (G1 * (np.sqrt((C1-x)**2 + C2) + (C1-x)) + 2*G2) / (F1 * (np.sqrt((C1-x)**2 + C2)) + F2 * (C1-x)+ 2*F3)


R_range = np.linspace(0, 1, 10000)

plt.plot(R_range, f(R_range))

plt.xlabel('Nuclear repressor protein level (R)')
plt.ylabel('Transcription rate')
plt.title('Figure S1')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


AT = 0.35
hot_beta = 0.01
cool_beta = 0.40
beta = cool_beta
sigma = 0.1
index_sigma = 1.0

## Comparison of the degree of equilibrium point shift with respect to temperature in uncoupled and coupled systems.

def coupled(Es, Ed, beta, mu):
  return AT*np.exp(3*(Es-Ed)*beta)*(1 - (1/index_sigma)*(mu / (np.exp(-Es*beta) + mu*(1+sigma**2))))

def coupled_ratio(Ep, Ed, hot_beta, cool_beta, hot_mu, cool_mu):
  return np.abs(coupled(Ep, Ed, cool_beta, cool_mu) - coupled(Ep, Ed, hot_beta, hot_mu))

Es_vals = [0.4]
Ed_vals = np.linspace(0.0, 0.4, 60)
Es, Ed = np.meshgrid(Es_vals, Ed_vals)

x = (Ed).flatten()

y_mu04 = coupled_ratio(Es, Ed, 0.01, 0.40, 0.4, 0.3).flatten()
y_mu0  = coupled_ratio(Es, Ed, 0.01, 0.40, 0.0, 0.0).flatten()

# Scatter plot
plt.figure(figsize=(8, 6))
plt.plot(x, y_mu04, color='gold')
plt.plot(x, y_mu0, color='gray')

plt.xlabel(r'$E_s - E_d$')
plt.ylabel('coupled value')
plt.title('Fig S2A')
plt.legend()
plt.show()

In [ ]:
## Calculating the coupled phase sensitivity

Es_value = [0.4]
Ed_value = np.linspace(0.0, 0.4, 10)
starting_coupling = 0.2

pulse_strength = 0.25
pulse_number = 1

Ed_list = []
coupled_phase_sensitivity_list = []

for Es, Ed in itertools.product(Es_value, Ed_value):

  hot_beta = 0.01
  cool_beta = 0.40

  print(f"Simulation for degradation energy : {Ed}")

  max_coupling = 2.0

  hot_coupling_strength = starting_coupling
  hot_success = False
  hot_signal = None

  while hot_coupling_strength <= max_coupling:
      hot_signal, hot_success = hot_particle_simulation(hot_coupling_strength, Es, Ed, hot_beta)

      if hot_success:
          print(f"Minimum coupling at: {hot_coupling_strength}")
          break

      else:
          print(f"Coupling does not occur at {hot_coupling_strength}")
          hot_coupling_strength += 0.05


  if not hot_success:
      print("Synchronization does not occur within this coupling range.")
      continue



  cool_coupling_strength = starting_coupling
  cool_success = False
  cool_signal = None


  while cool_coupling_strength <= max_coupling:
      cool_signal, cool_success = cool_particle_simulation(cool_coupling_strength, Es, Ed, cool_beta)

      if cool_success:
          print(f"Minimum coupling at: {cool_coupling_strength}")
          break

      else:
          print(f"Coupling does not occur at {cool_coupling_strength}")
          cool_coupling_strength += 0.05


  if not cool_success:
      print("Synchronization does not occur within this coupling range.")
      continue


  coupled_hot_signal = hot_signal
  coupled_cool_signal = cool_signal

  coupled_hot_period_list = np.array([period(coupled_hot_signal[:, 2, i]) for i in range(num_cell)])
  coupled_cool_period_list = np.array([period(coupled_cool_signal[:, 2, i]) for i in range(num_cell)])

  valid_condition = ((coupled_hot_period_list != None) & (coupled_cool_period_list != None))
  valid_indices = np.where(valid_condition)[0]

  valid_hot_period_list = coupled_hot_period_list[valid_condition]
  valid_cool_period_list = coupled_cool_period_list[valid_condition]

  if len(valid_hot_period_list) == 0 or len(valid_cool_period_list) == 0 :
    print(f"No valid period values found for both uncoupled and coupled systems")
    continue

  coupled_hot_period = np.mean(valid_hot_period_list)
  coupled_cool_period = np.mean(valid_cool_period_list)

  print(f"coupled hot period : {coupled_hot_period}")
  print(f"coupled cool period : {coupled_cool_period}")


  np.random.seed(40)
  index = int(np.random.choice(valid_indices, 1, replace=False))
  print(index)

  cool_peaks,_ = find_peaks(coupled_cool_signal[int(0.5 * step_count):, 2, index])

  pulse_time = int((pulse_strength) * (coupled_cool_period / step_size))
  pulse_number = 30

  starting_point = int(cool_peaks[1] + 0.5 * step_count)
  ending_point = int(cool_peaks[2] + 0.5 * step_count)

  pulse_points = np.linspace(starting_point, ending_point, pulse_number)
  phase_shift_list = []


  for pulse_point in pulse_points:

    perturbed_particle = np.zeros(((step_count - int(pulse_point) + 1), 4, num_cell))
    perturbed_particle[0, :, :] = coupled_cool_signal[int(pulse_point), :, :]

    for i in range(pulse_time + 1):
      V_sum = np.sum(perturbed_particle[i, 3, :])

      for j in range(num_cell):
            y = cell_culture[j]
            perturbed_particle[i+1, :, j] = hot_update(perturbed_particle[i, 0, j], perturbed_particle[i, 1, j], perturbed_particle[i, 2, j], perturbed_particle[i, 3, j], V_sum, y, hot_coupling_strength, Es, Ed, hot_beta)


    for k in range(step_count - int(pulse_point) - pulse_time - 1):
      V_sum = np.sum(perturbed_particle[k+1+pulse_time, 3, :])

      for j in range(num_cell):
       y = cell_culture[j]
       perturbed_particle[k+2+pulse_time, :, j] = cool_update(perturbed_particle[k+1+pulse_time, 0, j], perturbed_particle[k+1+pulse_time, 1, j], perturbed_particle[k+1+pulse_time, 2, j], perturbed_particle[k+1+pulse_time, 3, j], V_sum, y, cool_coupling_strength, Es, Ed, cool_beta)


    rescaled_signal = cool_signal[int(pulse_point):, 2, index]
    rescaled_time = time[int(pulse_point):]

    unperturbed_peak,_ = find_peaks(rescaled_signal)
    length_of_peak = len(unperturbed_peak)

    if length_of_peak < 2 :
      print("Not enough peak")

    one_peak_point = unperturbed_peak[length_of_peak - 2]
    other_peak_point = unperturbed_peak[length_of_peak - 1]

    perturbed_peak,_ = find_peaks(perturbed_particle[:, 2, index])
    perturbed_valley,_ = find_peaks(-perturbed_particle[:, 2, index])
    valley,_ = find_peaks(-rescaled_signal)

    valley_point = [x for x in valley if one_peak_point < x < other_peak_point]
    peak_perturbed_point = [x for x in perturbed_peak if one_peak_point < x < other_peak_point]
    valley_perturbed_point = [x for x in perturbed_valley if one_peak_point < x < other_peak_point]

    if len(peak_perturbed_point) != 1 or len(valley_perturbed_point) != 1 or len(valley_point) != 1 :

      phase_shift = 0

    else :

      phase_shift = 24 * ( (valley_point[0] - valley_perturbed_point[0]) / int((coupled_cool_period / step_size)) )

    phase_shift_list.append(phase_shift)

  maximal_shift = max(phase_shift_list)
  minimal_shift = min(phase_shift_list)
  phase_sensitivity = (maximal_shift - minimal_shift)/2
  Ed_list.append(Ed)
  coupled_phase_sensitivity_list.append(phase_sensitivity)

plt.scatter(Ed_list, coupled_phase_sensitivity_list, color='gold')
plt.plot(Ed_list, coupled_phase_sensitivity_list, color='gold')
plt.xlabel(r'$E_d$')
plt.ylabel('Phase sensitivity')
plt.title('Fig S3A')
plt.show()

In [ ]:
## Calculated results
Ed_list = np.linspace(0.0, 0.4, 10)
coupled_phase_sensitivity_list = [4.768, 3.82245430809399, 3.13043478260869, 2.61654135338346, 2.17647058823529, 1.75961538461539, 1.10117647058823, 0.855172413793103, 0.323595505617978, 0.210526315789474]
uncoupled_phase_sensitivity_list = [6.016085790884718, 4.875989445910291, 4.041450777202073, 3.267175572519083, 2.6100000000000003, 2.0294117647058822, 1.4388489208633093, 0.9577464788732395, 0.4413793103448276, 0.026966292134831482]
# These data was calculated in Fig 3A

plt.scatter(Ed_list, coupled_phase_sensitivity_list, color='gold')
plt.plot(Ed_list, coupled_phase_sensitivity_list, color='gold')
plt.scatter(Ed_list, uncoupled_phase_sensitivity_list, color='gray')
plt.plot(Ed_list, uncoupled_phase_sensitivity_list, color='gray')
plt.xlabel(r'$E_d$')
plt.ylabel('Phase sensitivity')
plt.title('Fig S3B')
plt.show()

In [ ]:
# Figure S3

In [ ]:
from scipy.integrate import solve_ivp

Es = 0.4
Ed = 0.1
AT = 0.35
initial_value = [0.1, 0.1, 0.1]

beta_value = np.linspace(0.0, 0.8, 9)  # temperature
Zf_integral_list = []
positive_phase_ratio = []
symmetry_list = []

step_size = 0.001
time_end = 50000
num_iter = int(time_end / step_size)

t_cycle = np.linspace(0, time_end, num_iter)

# Transcription function is too complicated, so we may use another function with similar shape.
def trans(z, alpha, k=200):
    return 1 / (1 + np.exp(k * (z / alpha - 1))) # alpha corresponds to AT (concentration of total activator)

def derivative(z, alpha, k=200):
    term = np.exp(k * (z / alpha - 1))
    return -k / alpha * term / ((1 + term)**2)

# Jacobian matrix
def jacobian_MR(x, y, z, alpha):
    dz = derivative(z, alpha)
    n = len(z)
    J = np.zeros((3, 3, n))

    J[0, 0, :] = -k2
    J[0, 1, :] = 0
    J[0, 2, :] = k1 * dz

    J[1, 0, :] = k1
    J[1, 1, :] = -k2
    J[1, 2, :] = 0

    J[2, 0, :] = 0
    J[2, 1, :] = k1
    J[2, 2, :] = -k2

    return J


def multiple_repressor(t, w):
    x, y, z = w
    dxdt = k1 * trans(z, AT) - k2 * x
    dydt = k1 * x - k2 * y
    dzdt = k1 * y - k2 * z
    return [dxdt, dydt, dzdt]


# Backward Euler method

def backward_solver(initial_data):

    x = np.zeros(len(t_cut))
    y = np.zeros(len(t_cut))
    z = np.zeros(len(t_cut))

    x[-1], y[-1], z[-1] = initial_data

    dt_scalar = t_cut[1] - t_cut[0]

    for i in range(len(t_cut) - 1, 0, -1):

        f = -J[:, :, i].T @ np.array([x[i], y[i], z[i]])
        x[i-1] = x[i] - dt_scalar * f[0]
        y[i-1] = y[i] - dt_scalar * f[1]
        z[i-1] = z[i] - dt_scalar * f[2]

    return x, y, z


for beta in beta_value:
    k1 = np.exp(-Es * beta)
    k2 = np.exp(-Ed * beta)

    sol = solve_ivp(
        multiple_repressor,
        [0, time_end],
        initial_value,
        t_eval=t_cycle,
        rtol=1e-6,
        atol=1e-9
    )

    x_cycle_full, y_cycle_full, z_cycle_full = sol.y

    cut = 40000
    start_idx = max(0, len(x_cycle_full) - cut)

    x_cycle = x_cycle_full[start_idx:]
    y_cycle = y_cycle_full[start_idx:]
    z_cycle = z_cycle_full[start_idx:]
    t_cut = t_cycle[start_idx:]

    dt = t_cut[1] - t_cut[0]

    J = jacobian_MR(x_cycle, y_cycle, z_cycle, AT)

    dx_dt = np.gradient(x_cycle, t_cut)
    dy_dt = np.gradient(y_cycle, t_cut)
    dz_dt = np.gradient(z_cycle, t_cut)

    np.random.seed(0)
    Zx_raw, Zy_raw, Zz_raw = backward_solver(np.random.rand(3))

    dot_product = (
        Zx_raw[-1] * dx_dt[-1] +
        Zy_raw[-1] * dy_dt[-1] +
        Zz_raw[-1] * dz_dt[-1]
    )

    if abs(dot_product) < 1e-9:
        norm_factor = 1.0
    else:
        norm_factor = dot_product

    Zx = Zx_raw / norm_factor
    Zy = Zy_raw / norm_factor
    Zz = Zz_raw / norm_factor

    peaks, _ = find_peaks(-Zx)

    if len(peaks) < 4:
        print(f"beta = {beta:.3f}: Not enough peaks found (len={len(peaks)}). Skipping.")

    i0, i1 = peaks[2], peaks[3]

    Z_cycle = Zx[i0:i1]
    z_cycle_local = z_cycle[i0:i1]
    x_cycle_local = x_cycle[i0:i1]

    t_local = t_cut[i0:i1]

    # Calculating convolution
    f_z = trans(z_cycle_local, AT)
    Zf_integral = np.trapz(Z_cycle * f_z, t_local)
    Zf_integral_list.append(Zf_integral)

    print(f"beta = {beta:.3f}, integral = {Zf_integral:.6f}")

    # Advance zone ratio
    positive_ratio = np.sum(Z_cycle > 0) / len(Z_cycle) * 100
    positive_phase_ratio.append(positive_ratio)

    # Rising ratio
    sym_val = symmetry(x_cycle)
    symmetry_list.append(sym_val)

beta_value = np.array(beta_value)
Zf_integral_list = np.array(Zf_integral_list)
valid_idx = ~np.isnan(Zf_integral_list)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(beta_value[valid_idx], np.array(positive_phase_ratio)[valid_idx], 'o-', lw=2, color='black')
plt.xlabel(r'$\beta$')
plt.ylabel('Positive phase ratio')
plt.title('Fig S4a-(i)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(beta_value[valid_idx], Zf_integral_list[valid_idx], 'o-', lw=2, color='black')
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\int_0^T Z(t)f(z(t))dt$')
plt.title('Fig S4a-(ii)')
plt.tight_layout()
plt.show()

In [ ]:
# Coupled system simulation

num_cell = 100
timescale = 20

np.random.seed(1)
cell_culture = np.random.normal(1, 0.1, num_cell)

def hot_update(M, Rc, R, V, V_sum, y, coupling_input, Ep_value, Ed_value, hot_temp):
  h = step_size
  k1 = np.exp(-Ep_value * hot_temp)
  k2 = np.exp(-Ed_value * hot_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h + (coupling_input / num_cell) * V_sum * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
      V + timescale * (f(R) - V) * h
  ])

def cool_update(M, Rc, R, V, V_sum, y, coupling_input, Ep_value, Ed_value, cool_temp):
  h = step_size
  k1 = np.exp(-Ep_value * cool_temp)
  k2 = np.exp(-Ed_value * cool_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h + (coupling_input / num_cell) * V_sum * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
      V + timescale * (f(R) - V) * h
  ])

def hot_update_uncoupled(M, Rc, R, y, Ep_value, Ed_value, hot_temp):
  h = step_size
  k1 = np.exp(-Ep_value * hot_temp)
  k2 = np.exp(-Ed_value * hot_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
  ])

def cool_update_uncoupled(M, Rc, R, y, Ep_value, Ed_value, cool_temp):
  h = step_size
  k1 = np.exp(-Ep_value * cool_temp)
  k2 = np.exp(-Ed_value * cool_temp)

  return np.array([
      M + (k1 * f(R) - k2 * M) * y * h,
      Rc + (k1 * M - k2 * Rc) * y * h,
      R + (k1 * Rc - k2 * R) * y * h,
  ])

def hot_particle_simulation(coupling_input, Ep_value, Ed_value, hot_temp):

    particle = np.zeros((step_count + 1, 4, num_cell))
    np.random.seed(2)
    particle[0, :, :] = np.random.uniform(0, 1, (4, num_cell))

    for i in range(step_count):
        V_sum = np.sum(particle[i, 3, :])

        for j in range(num_cell):
            y = cell_culture[j]
            particle[i + 1, :, j] = hot_update(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], particle[i, 3, j], V_sum, y, coupling_input, Ep_value, Ed_value, hot_temp)

    freqs = np.array([frequency(particle[:, 2, n]) for n in range(num_cell)])
    valid_freqs = freqs[freqs != None]

    if len(valid_freqs) == 0:
        return None, False

    mean_freq = np.mean(valid_freqs)
    filtered_valid_freqs = valid_freqs[np.abs(valid_freqs - mean_freq) < 0.05 * 0.1]

    if (len(filtered_valid_freqs) / num_cell) < 0.95:
        return None, False

    else:
        return particle, True


def cool_particle_simulation(coupling_input, Ep_value, Ed_value, cool_temp):
    particle = np.zeros((step_count + 1, 4, num_cell))
    np.random.seed(2)
    particle[0, :, :] = np.random.uniform(0, 1, (4, num_cell))

    for i in range(step_count):
        V_sum = np.sum(particle[i, 3, :])

        for j in range(num_cell):
            y = cell_culture[j]
            particle[i + 1, :, j] = cool_update(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], particle[i, 3, j], V_sum, y, coupling_input, Ep_value, Ed_value, cool_temp)

    freqs = np.array([frequency(particle[:, 2, n]) for n in range(num_cell)])
    valid_freqs = freqs[freqs != None]

    if len(valid_freqs) == 0:
        return None, False

    mean_freq = np.mean(valid_freqs)
    filtered_valid_freqs = valid_freqs[np.abs(valid_freqs - mean_freq) < 0.05*0.1]

    if (len(filtered_valid_freqs) / num_cell) < 0.95:
        return None, False

    else:
        return particle, True


def hot_particle_simulation_uncoupled(Ep_value, Ed_value, hot_temp) :
  particle = np.zeros((step_count + 1, 3, num_cell))
  np.random.seed(2)
  particle[0, :, :] = np.random.uniform(0, 1, (3, num_cell))

  for j in range(num_cell):
    y = cell_culture[j]

    for i in range(step_count):
      particle[i+1, :, j] = hot_update_uncoupled(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], y, Ep_value, Ed_value, hot_temp)

  return particle


def cool_particle_simulation_uncoupled(Ep_value, Ed_value, cool_temp) :
  particle = np.zeros((step_count + 1, 3, num_cell))
  np.random.seed(2)
  particle[0, :, :] = np.random.uniform(0, 1, (3, num_cell))

  for j in range(num_cell):
    y = cell_culture[j]

    for i in range(step_count):
      particle[i+1, :, j] = cool_update_uncoupled(particle[i, 0, j], particle[i, 1, j], particle[i, 2, j], y, Ep_value, Ed_value, cool_temp)

  return particle

In [ ]:
Es = 0.4
Ed = 0.1
starting_coupling = 0.2
pulse_strength = 0.25
pulse_number = 30
coupled_period_list = []
uncoupled_period_list = []
gap_list = []

max_coupling = 0.8
beta_list = np.linspace(0.0, 0.8, 9)

for beta in beta_list:

 test_coupling_strength = starting_coupling
 test_success = False
 test_signal = None

 while test_coupling_strength <= max_coupling:
     test_signal, test_success = hot_particle_simulation(test_coupling_strength, Es, Ed, beta)

     if test_success:
         print(f"Minimum coupling at: {test_coupling_strength}")
         break

     else:
         print(f"Coupling does not occur at {test_coupling_strength}")
         test_coupling_strength += 0.05

 coupled_test_signal = test_signal
 uncoupled_test_signal = hot_particle_simulation_uncoupled(Es, Ed, beta)

 coupled_test_period_list = np.array([period(coupled_test_signal[:, 2, i]) for i in range(num_cell)])
 uncoupled_test_period_list = np.array([period(uncoupled_test_signal[:, 2, i]) for i in range(num_cell)])

 valid_condition = ((coupled_test_period_list != None))
 valid_indices = np.where(valid_condition)[0]

 valid_test_period_list = coupled_test_period_list[valid_condition]
 valid_uncoupled_test_period_list = uncoupled_test_period_list[valid_condition]

 coupled_test_period = np.mean(valid_test_period_list)
 uncoupled_test_period = np.mean(valid_uncoupled_test_period_list)

 coupled_period_list.append(coupled_test_period)
 uncoupled_period_list.append(uncoupled_test_period)
 gap_list.append((coupled_test_period - uncoupled_test_period)/(uncoupled_test_period))

In [ ]:
### The period values need to be rescaled!

plt.figure(figsize=(6,4))
plt.scatter(beta_list, uncoupled_period_list, color='gray')
plt.scatter(beta_list, coupled_period_list, color='gold')
plt.plot(beta_list, uncoupled_period_list, color='gray')
plt.plot(beta_list, coupled_period_list, color='gold')

plt.xlabel(r'beta')
plt.ylabel('Period (hr)')
plt.title('Fig S4b-(i)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(beta_list, gap_list, color='black')
plt.plot(beta_list, gap_list, color='black')

plt.xlabel(r'beta')
plt.ylabel('Normalized difference of peirod')
plt.title('Fig S4b-(ii)')
plt.tight_layout()
plt.show()